# Evaluation of precomputed full-time-series archives

This notebook compares stored GT, standard M3C2, and TAM3C2 4D-OBC results. It uses continuous timestamp durations, evaluates global change support and object-level one-to-one matching, diagnoses overlap topology, and finishes with optional individual-object inspection. Distance estimation and object extraction are not rerun here.


In [ ]:
%load_ext autoreload
%autoreload 2

import glob
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py4dgeo

from matplotlib.patches import Polygon
from matplotlib.ticker import MaxNLocator
from scipy.optimize import linear_sum_assignment
from scipy.spatial import ConvexHull, QhullError

METHODS = ("M3C2", "TAM3C2")
METHOD_COLORS = {"M3C2": "#4C78A8", "TAM3C2": "#F58518"}
IOU_EPS = 1e-3
_IOU_TOL = IOU_EPS


## 1. Configuration

Set the archive locations and the formal object-match threshold. The topology threshold controls which positive ST-IoU relations count as graph edges; use a larger value such as 0.01 or 0.05 only for sensitivity tests.


In [ ]:
archive_folder = os.path.join(os.getcwd(), "simulation_test2_als_downsampled1")
tam3c2_archive_spec = "*_full_timeseries_best_scale_idx*_weighted_tam3c2.zip"
m3c2_archive_spec = "*_full_timeseries_best_scale_idx*_standard_m3c2.zip"
reference_archive_path = os.path.join(os.getcwd(), "simulation2_reference.zip")

# Temporal-gap alternatives:
# tam3c2_archive_spec = "*_full_timeline_reconstructed_best_scale_idx*_weighted_tam3c2.zip"
# m3c2_archive_spec = "*_observed_epochs_only_best_scale_idx*_standard_m3c2.zip"

match_iou_threshold = 0.80
topology_iou_threshold = IOU_EPS
reference_order_mode = "reference_index"  # or "tam3c2_descending"


## 2. Load precomputed archives

Resolve exactly one archive for each method, load the stored analyses, and verify that object results are available. All later sections use the single `analyses` dictionary created here.


In [ ]:
def resolve_archive(folder, archive_spec, label):
    folder = os.path.abspath(folder)
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Archive folder not found: {folder}")

    candidate = (
        archive_spec
        if os.path.isabs(archive_spec)
        else os.path.join(folder, archive_spec)
    )

    if glob.has_magic(candidate):
        matches = sorted(glob.glob(candidate))

        # If the specified folder is a parent directory, also search below it.
        if not matches and not os.path.isabs(archive_spec):
            recursive_pattern = os.path.join(folder, "**", archive_spec)
            matches = sorted(glob.glob(recursive_pattern, recursive=True))

        matches = [match for match in matches if os.path.isfile(match)]
        if len(matches) != 1:
            available_zip_files = sorted(
                glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True)
            )
            available_text = "\n".join(
                f"  - {zip_path}" for zip_path in available_zip_files
            ) or "  (no ZIP files found)"
            raise ValueError(
                f"{label} archive pattern must match exactly one file; "
                f"found {len(matches)} matches for {archive_spec!r} under {folder!r}.\n"
                f"Available ZIP files:\n{available_text}"
            )
        candidate = matches[0]

    if not os.path.isfile(candidate):
        available_zip_files = sorted(
            glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True)
        )
        available_text = "\n".join(
            f"  - {zip_path}" for zip_path in available_zip_files
        ) or "  (no ZIP files found)"
        raise FileNotFoundError(
            f"{label} archive not found: {candidate}\n"
            f"Available ZIP files under {folder}:\n{available_text}"
        )
    return os.path.abspath(candidate)

gt_path = os.path.abspath(reference_archive_path)
if not os.path.isfile(gt_path):
    raise FileNotFoundError(f"Reference archive not found: {gt_path}")

tam3c2_path = resolve_archive(
    archive_folder,
    tam3c2_archive_spec,
    "TAM3C2",
)
m3c2_path = resolve_archive(
    archive_folder,
    m3c2_archive_spec,
    "M3C2",
)

ref_analysis = py4dgeo.SpatiotemporalAnalysis(gt_path, force=False)
analysis = py4dgeo.SpatiotemporalAnalysis(tam3c2_path, force=False)
m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(m3c2_path, force=False)

analyses = {
    "GT": ref_analysis,
    "M3C2": m3c2_analysis,
    "TAM3C2": analysis,
}

for name, st_analysis in analyses.items():
    if st_analysis.objects is None:
        raise RuntimeError(
            f"{name} archive contains no stored 4D-OBC objects: "
            f"{st_analysis.filename}. Run object extraction in the producer notebook first."
        )
    print(
        f"{name}: {len(st_analysis.objects)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


## 3. Continuous-support evaluation

Each object is represented by its core-point set and continuous timestamp interval. Pairwise spatial, temporal, and spatiotemporal IoUs are computed for every detected-reference combination. Formal object matches use a global Hungarian one-to-one assignment followed by the ST-IoU threshold; global metrics merge all detected supports and ignore object identity.


In [ ]:
def _timestamps_for_analysis(st_analysis):
    timedeltas = list(st_analysis.timedeltas)
    if len(timedeltas) != st_analysis.distances.shape[1]:
        raise ValueError(
            f"{st_analysis.filename}: {len(timedeltas)} timestamps for "
            f"{st_analysis.distances.shape[1]} distance epochs"
        )

    reference_time = st_analysis.reference_epoch.timestamp
    timestamps = np.array([reference_time + td for td in timedeltas], dtype=object)
    if len(timestamps) == 0:
        raise ValueError(f"{st_analysis.filename}: no acquisition timestamps available")

    time_days = np.array(
        [(ts - timestamps[0]).total_seconds() / 86400.0 for ts in timestamps],
        dtype=float,
    )
    return timestamps, time_days


def _validate_common_corepoints(analyses):
    base_name = 'GT'
    base_corepoints = np.asarray(analyses[base_name].corepoints.cloud)
    for name, st_analysis in analyses.items():
        corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
        if corepoints_for_method.shape != base_corepoints.shape or not np.allclose(
            corepoints_for_method,
            base_corepoints,
        ):
            raise ValueError(
                f"{name} corepoints differ from {base_name}; this evaluation assumes "
                "identical core-point arrays and indices."
            )
    print(
        f"All analyses use the same {base_corepoints.shape[0]:,} corepoints; "
        "nearest-neighbor corepoint mapping is bypassed."
    )


def _object_properties(obj, time_days):
    corepoints_for_object = set(int(idx) for idx in np.asarray(obj.indices, dtype=int))
    start_epoch = int(obj.start_epoch)
    end_epoch = int(obj.end_epoch)

    if start_epoch > end_epoch:
        raise ValueError(f"Invalid object interval: {start_epoch} > {end_epoch}")
    if start_epoch < 0 or end_epoch >= len(time_days):
        raise IndexError(
            f"Object epoch interval [{start_epoch}, {end_epoch}] outside "
            f"timestamp array of length {len(time_days)}"
        )
    if not corepoints_for_object:
        raise ValueError("Object has empty corepoint support")

    start_time = float(time_days[start_epoch])
    end_time = float(time_days[end_epoch])
    duration = end_time - start_time
    if duration < 0:
        raise ValueError(f"Object has negative duration: {duration}")

    return {
        'corepoints': corepoints_for_object,
        'start_epoch': start_epoch,
        'end_epoch': end_epoch,
        'start_time': start_time,
        'end_time': end_time,
        'duration': duration,
        'n_corepoints': len(corepoints_for_object),
    }


def _pairwise_object_metrics(det_obj, ref_obj, det_time_days, ref_time_days):
    det = _object_properties(det_obj, det_time_days)
    ref = _object_properties(ref_obj, ref_time_days)

    shared_corepoint_count = len(det['corepoints'] & ref['corepoints'])
    spatial_union_count = det['n_corepoints'] + ref['n_corepoints'] - shared_corepoint_count
    spatial_iou = (
        shared_corepoint_count / spatial_union_count
        if spatial_union_count > 0
        else np.nan
    )

    overlapping_duration = max(
        0.0,
        min(det['end_time'], ref['end_time'])
        - max(det['start_time'], ref['start_time']),
    )
    temporal_union = det['duration'] + ref['duration'] - overlapping_duration
    temporal_iou = overlapping_duration / temporal_union if temporal_union > 0 else np.nan

    detected_st_support = det['n_corepoints'] * det['duration']
    reference_st_support = ref['n_corepoints'] * ref['duration']
    intersection_st_support = shared_corepoint_count * overlapping_duration
    union_st_support = detected_st_support + reference_st_support - intersection_st_support
    spatiotemporal_iou = (
        intersection_st_support / union_st_support
        if union_st_support > 0
        else np.nan
    )

    if np.isfinite(spatiotemporal_iou):
        if np.isfinite(spatial_iou) and spatiotemporal_iou > spatial_iou + _IOU_TOL:
            raise AssertionError("Spatiotemporal IoU exceeds spatial IoU")
        if np.isfinite(temporal_iou) and spatiotemporal_iou > temporal_iou + _IOU_TOL:
            raise AssertionError("Spatiotemporal IoU exceeds temporal IoU")

    return {
        'detected_corepoint_count': det['n_corepoints'],
        'reference_corepoint_count': ref['n_corepoints'],
        'shared_corepoint_count': shared_corepoint_count,
        'detected_start_time': det['start_time'],
        'detected_end_time': det['end_time'],
        'reference_start_time': ref['start_time'],
        'reference_end_time': ref['end_time'],
        'detected_duration': det['duration'],
        'reference_duration': ref['duration'],
        'overlapping_duration': overlapping_duration,
        'spatial_iou': spatial_iou,
        'temporal_iou': temporal_iou,
        'spatiotemporal_iou': spatiotemporal_iou,
        'detected_spatiotemporal_support': detected_st_support,
        'reference_spatiotemporal_support': reference_st_support,
        'intersection_spatiotemporal_support': intersection_st_support,
        'union_spatiotemporal_support': union_st_support,
    }


def _pairwise_metrics_table(method, detected_objects, reference_objects, det_time_days, ref_time_days):
    rows = []
    for detected_idx, det_obj in enumerate(detected_objects):
        for reference_idx, ref_obj in enumerate(reference_objects):
            row = {
                'method': method,
                'detected_object_index': detected_idx,
                'reference_object_index': reference_idx,
            }
            row.update(_pairwise_object_metrics(det_obj, ref_obj, det_time_days, ref_time_days))
            rows.append(row)
    return rows


def _spatiotemporal_iou_matrix(pair_rows, n_detected, n_reference):
    matrix = np.full((n_detected, n_reference), np.nan, dtype=float)
    for row in pair_rows:
        matrix[row['detected_object_index'], row['reference_object_index']] = row[
            'spatiotemporal_iou'
        ]
    return matrix


def _one_to_one_assignment(spatiotemporal_iou_matrix):
    """Return the total-ST-IoU-maximizing one-to-one assignment."""
    if 0 in spatiotemporal_iou_matrix.shape:
        return []
    cost = np.where(
        np.isfinite(spatiotemporal_iou_matrix),
        1.0 - spatiotemporal_iou_matrix,
        1e6,
    )
    detected_indices, reference_indices = linear_sum_assignment(cost)
    return [
        (int(detected_idx), int(reference_idx))
        for detected_idx, reference_idx in zip(detected_indices, reference_indices)
        if np.isfinite(spatiotemporal_iou_matrix[detected_idx, reference_idx])
    ]


def _one_to_one_matches(spatiotemporal_iou_matrix, threshold):
    return [
        (detected_idx, reference_idx)
        for detected_idx, reference_idx in _one_to_one_assignment(
            spatiotemporal_iou_matrix
        )
        if spatiotemporal_iou_matrix[detected_idx, reference_idx] >= threshold
    ]


def _merge_intervals(intervals):
    merged = []
    for start_time, end_time in sorted(intervals):
        if not np.isfinite(start_time) or not np.isfinite(end_time):
            continue
        if end_time < start_time:
            raise ValueError(f"Invalid interval [{start_time}, {end_time}]")
        if not merged or start_time > merged[-1][1] + _IOU_TOL:
            merged.append([float(start_time), float(end_time)])
        else:
            merged[-1][1] = max(merged[-1][1], float(end_time))
    return [(start_time, end_time) for start_time, end_time in merged]


def _intervals_by_corepoint(objects_for_method, time_days):
    intervals = {}
    for obj in objects_for_method:
        props = _object_properties(obj, time_days)
        if props['duration'] == 0:
            continue
        interval = (props['start_time'], props['end_time'])
        for corepoint_idx in props['corepoints']:
            intervals.setdefault(corepoint_idx, []).append(interval)
    return {
        corepoint_idx: _merge_intervals(corepoint_intervals)
        for corepoint_idx, corepoint_intervals in intervals.items()
    }


def _total_interval_duration(intervals):
    return float(sum(end_time - start_time for start_time, end_time in intervals))


def _shared_interval_duration(left_intervals, right_intervals):
    total = 0.0
    left_idx = 0
    right_idx = 0
    while left_idx < len(left_intervals) and right_idx < len(right_intervals):
        left_start, left_end = left_intervals[left_idx]
        right_start, right_end = right_intervals[right_idx]
        total += max(0.0, min(left_end, right_end) - max(left_start, right_start))
        if left_end <= right_end:
            left_idx += 1
        else:
            right_idx += 1
    return total


def _global_support_metrics(detected_objects, reference_objects, det_time_days, ref_time_days):
    detected_by_corepoint = _intervals_by_corepoint(detected_objects, det_time_days)
    reference_by_corepoint = _intervals_by_corepoint(reference_objects, ref_time_days)
    corepoint_indices = set(detected_by_corepoint) | set(reference_by_corepoint)

    tp = 0.0
    detected_total = 0.0
    reference_total = 0.0
    for corepoint_idx in corepoint_indices:
        detected_intervals = detected_by_corepoint.get(corepoint_idx, [])
        reference_intervals = reference_by_corepoint.get(corepoint_idx, [])
        detected_total += _total_interval_duration(detected_intervals)
        reference_total += _total_interval_duration(reference_intervals)
        tp += _shared_interval_duration(detected_intervals, reference_intervals)

    fp = max(0.0, detected_total - tp)
    fn = max(0.0, reference_total - tp)
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = tp / (tp + fn) if tp + fn > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan
    iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else np.nan

    return {
        'global_tp': tp,
        'global_fp': fp,
        'global_fn': fn,
        'global_precision': precision,
        'global_recall': recall,
        'global_f1': f1,
        'global_iou': iou,
    }


def _trapezoid(values, times):
    if hasattr(np, 'trapezoid'):
        return float(np.trapezoid(values, times))
    return float(np.trapz(values, times))


def _mean_time_integrated_change(obj, distances, time_days):
    props = _object_properties(obj, time_days)
    epoch_slice = slice(props['start_epoch'], props['end_epoch'] + 1)
    object_times = time_days[epoch_slice] - time_days[props['start_epoch']]
    if len(object_times) < 2:
        return {
            'change_volume': np.nan,
            'valid_corepoint_count': 0,
            'valid_corepoint_percentage': 0.0,
        }

    integrated_changes = []
    for corepoint_idx in sorted(props['corepoints']):
        series = np.asarray(distances[corepoint_idx, epoch_slice], dtype=float)
        if not np.isfinite(series[0]):
            continue
        finite = np.isfinite(series)
        if finite.sum() < 2:
            continue
        change = np.abs(series[finite] - series[0])
        integrated_changes.append(_trapezoid(change, object_times[finite]))

    valid_count = len(integrated_changes)
    return {
        'change_volume': float(np.mean(integrated_changes)) if valid_count else np.nan,
        'valid_corepoint_count': valid_count,
        'valid_corepoint_percentage': 100.0 * valid_count / props['n_corepoints'],
    }


def _summary_stats(values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {
            f'mean_{prefix}': np.nan,
            f'median_{prefix}': np.nan,
            f'std_{prefix}': np.nan,
            f'min_{prefix}': np.nan,
            f'max_{prefix}': np.nan,
        }
    return {
        f'mean_{prefix}': float(np.mean(values)),
        f'median_{prefix}': float(np.median(values)),
        f'std_{prefix}': float(np.std(values)),
        f'min_{prefix}': float(np.min(values)),
        f'max_{prefix}': float(np.max(values)),
    }


def _harmonic_mean(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan


def _evaluate_method(method, method_analysis, gt_analysis, timestamp_info):
    detected_objects = list(method_analysis.objects)
    reference_objects = list(gt_analysis.objects)
    detected_timestamps, detected_time_days = timestamp_info[method]
    reference_timestamps, reference_time_days = timestamp_info['GT']

    pair_rows = _pairwise_metrics_table(
        method,
        detected_objects,
        reference_objects,
        detected_time_days,
        reference_time_days,
    )
    pair_lookup = {
        (row['detected_object_index'], row['reference_object_index']): row
        for row in pair_rows
    }
    iou_matrix = _spatiotemporal_iou_matrix(
        pair_rows,
        len(detected_objects),
        len(reference_objects),
    )
    matched_pairs = _one_to_one_matches(iou_matrix, match_iou_threshold)

    matched_rows = []
    for detected_idx, reference_idx in matched_pairs:
        pair_row = pair_lookup[(detected_idx, reference_idx)]
        detected_change = _mean_time_integrated_change(
            detected_objects[detected_idx],
            method_analysis.distances,
            detected_time_days,
        )
        reference_change = _mean_time_integrated_change(
            reference_objects[reference_idx],
            gt_analysis.distances,
            reference_time_days,
        )
        change_difference = detected_change['change_volume'] - reference_change['change_volume']
        absolute_change_difference = abs(change_difference)

        matched_rows.append({
            'method': method,
            'detected_object_index': detected_idx,
            'reference_object_index': reference_idx,
            'spatial_iou': pair_row['spatial_iou'],
            'temporal_iou': pair_row['temporal_iou'],
            'spatiotemporal_iou': pair_row['spatiotemporal_iou'],
            'detected_corepoint_count': pair_row['detected_corepoint_count'],
            'reference_corepoint_count': pair_row['reference_corepoint_count'],
            'shared_corepoint_count': pair_row['shared_corepoint_count'],
            'detected_start_timestamp': detected_timestamps[int(detected_objects[detected_idx].start_epoch)],
            'detected_end_timestamp': detected_timestamps[int(detected_objects[detected_idx].end_epoch)],
            'reference_start_timestamp': reference_timestamps[int(reference_objects[reference_idx].start_epoch)],
            'reference_end_timestamp': reference_timestamps[int(reference_objects[reference_idx].end_epoch)],
            'detected_duration_days': pair_row['detected_duration'],
            'reference_duration_days': pair_row['reference_duration'],
            'overlapping_duration_days': pair_row['overlapping_duration'],
            'detected_change_volume': detected_change['change_volume'],
            'reference_change_volume': reference_change['change_volume'],
            'change_volume_difference': change_difference,
            'absolute_change_volume_difference': absolute_change_difference,
            'detected_valid_corepoint_count': detected_change['valid_corepoint_count'],
            'reference_valid_corepoint_count': reference_change['valid_corepoint_count'],
            'detected_valid_corepoint_percentage': detected_change['valid_corepoint_percentage'],
            'reference_valid_corepoint_percentage': reference_change['valid_corepoint_percentage'],
        })

    n_detected = len(detected_objects)
    n_reference = len(reference_objects)
    n_matched = len(matched_rows)
    object_precision = n_matched / n_detected if n_detected else np.nan
    object_recall = n_matched / n_reference if n_reference else np.nan
    object_f1 = _harmonic_mean(object_precision, object_recall)

    summary = {
        'method': method,
        'n_detected_objects': n_detected,
        'n_reference_objects': n_reference,
        'n_matched_pairs': n_matched,
        'n_unmatched_detected_objects': n_detected - n_matched,
        'n_unmatched_reference_objects': n_reference - n_matched,
        'object_precision': object_precision,
        'object_recall': object_recall,
        'object_f1': object_f1,
    }
    summary.update(_global_support_metrics(detected_objects, reference_objects, detected_time_days, reference_time_days))

    for column in ['spatial_iou', 'temporal_iou', 'spatiotemporal_iou']:
        summary.update(_summary_stats([row[column] for row in matched_rows], column))

    for column in ['change_volume_difference', 'absolute_change_volume_difference']:
        stats = _summary_stats([row[column] for row in matched_rows], column)
        summary[f'mean_{column}'] = stats[f'mean_{column}']
        summary[f'median_{column}'] = stats[f'median_{column}']

    return summary, matched_rows, pair_rows


In [ ]:
_validate_common_corepoints(analyses)
timestamp_info = {
    name: _timestamps_for_analysis(st_analysis)
    for name, st_analysis in analyses.items()
}

base_timestamps = timestamp_info["GT"][0]
for name, (timestamps, _) in timestamp_info.items():
    if len(timestamps) != len(base_timestamps) or any(
        timestamp != base_timestamp
        for timestamp, base_timestamp in zip(timestamps, base_timestamps)
    ):
        raise ValueError(f"{name} timestamps differ from GT timestamps")

summary_rows, matched_rows, pairwise_rows = [], [], []
for method in METHODS:
    summary, method_matches, method_pairs = _evaluate_method(
        method,
        analyses[method],
        analyses["GT"],
        timestamp_info,
    )
    summary_rows.append(summary)
    matched_rows.extend(method_matches)
    pairwise_rows.extend(method_pairs)

summary_df = pd.DataFrame(summary_rows).set_index("method")
pairwise_metrics_df = pd.DataFrame(pairwise_rows)
matched_objects_df = pd.DataFrame(matched_rows)
if matched_objects_df.empty:
    matched_objects_df = pd.DataFrame(
        columns=[
            "method",
            "detected_object_index",
            "reference_object_index",
            "spatial_iou",
            "temporal_iou",
            "spatiotemporal_iou",
        ]
    )

summary_columns = [
    "n_detected_objects", "n_reference_objects", "n_matched_pairs",
    "n_unmatched_detected_objects", "n_unmatched_reference_objects",
    "object_precision", "object_recall", "object_f1",
    "global_tp", "global_fp", "global_fn",
    "global_precision", "global_recall", "global_f1", "global_iou",
    "mean_spatial_iou", "mean_temporal_iou", "mean_spatiotemporal_iou",
    "mean_absolute_change_volume_difference",
]
display(summary_df[summary_columns])
display(
    matched_objects_df.sort_values(
        ["method", "spatiotemporal_iou"],
        ascending=[True, False],
    )
)


## 4. Overall performance

The first chart evaluates total continuous change support without object identities. The second evaluates successfully recovered independent objects after strict one-to-one matching. Read the two together: high global agreement with weaker object scores indicates correct support but imperfect delineation.


In [ ]:
def plot_metric_bars(columns, labels, title):
    ax = summary_df[list(columns)].rename(columns=dict(zip(columns, labels))).plot(
        kind="bar", figsize=(9, 4), ylim=(0, 1), rot=0
    )
    ax.set(title=title, ylabel="Score")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_metric_bars(
    ("global_precision", "global_recall", "global_f1", "global_iou"),
    ("Precision", "Recall", "F1", "IoU"),
    "Global continuous-support agreement against GT",
)
plot_metric_bars(
    ("object_precision", "object_recall", "object_f1"),
    ("Object precision", "Object recall", "Object F1"),
    "Strict one-to-one object detection metrics",
)


## 5. GT-centred one-to-one comparison

Hungarian assignment is performed separately for M3C2 and TAM3C2, then results are aligned by the real reference-object index. Vertical connectors compare the two methods on the same GT object. Points below the dashed threshold are assigned pairs but not formal successful matches. The default index order is the most transparent; TAM3C2-based ordering is available only as a diagnostic view.


In [ ]:
n_reference_objects = len(analyses["GT"].objects)
reference_indices = np.arange(n_reference_objects)
assignment_rows = []

for method in METHODS:
    method_pairs = pairwise_metrics_df.query("method == @method")
    matrix = _spatiotemporal_iou_matrix(
        method_pairs.to_dict("records"),
        len(analyses[method].objects),
        n_reference_objects,
    )
    pair_lookup = {
        (int(row.detected_object_index), int(row.reference_object_index)): row
        for row in method_pairs.itertuples(index=False)
    }
    assigned_by_reference = {
        reference_idx: detected_idx
        for detected_idx, reference_idx in _one_to_one_assignment(matrix)
    }

    for reference_idx in reference_indices:
        detected_idx = assigned_by_reference.get(int(reference_idx))
        if detected_idx is None:
            assignment_rows.append({
                "method": method,
                "reference_object_index": int(reference_idx),
                "detected_object_index": np.nan,
                "spatial_iou": np.nan,
                "temporal_iou": np.nan,
                "spatiotemporal_iou": np.nan,
                "successful": False,
            })
            continue

        pair = pair_lookup[(detected_idx, int(reference_idx))]
        st_iou = float(pair.spatiotemporal_iou)
        assignment_rows.append({
            "method": method,
            "reference_object_index": int(reference_idx),
            "detected_object_index": detected_idx,
            "spatial_iou": float(pair.spatial_iou),
            "temporal_iou": float(pair.temporal_iou),
            "spatiotemporal_iou": st_iou,
            "successful": st_iou >= match_iou_threshold,
        })

gt_assignment_df = pd.DataFrame(assignment_rows)
iou_pivot = gt_assignment_df.pivot(
    index="reference_object_index", columns="method", values="spatiotemporal_iou"
).reindex(index=reference_indices, columns=METHODS)
success_pivot = gt_assignment_df.pivot(
    index="reference_object_index", columns="method", values="successful"
).reindex(index=reference_indices, columns=METHODS).fillna(False).astype(bool)

m3c2_iou = iou_pivot["M3C2"].to_numpy(float)
tam3c2_iou = iou_pivot["TAM3C2"].to_numpy(float)
m3c2_success = success_pivot["M3C2"].to_numpy(bool)
tam3c2_success = success_pivot["TAM3C2"].to_numpy(bool)

if reference_order_mode == "reference_index":
    order = reference_indices
    xlabel = "Reference-object index"
elif reference_order_mode == "tam3c2_descending":
    order = np.argsort(-np.nan_to_num(tam3c2_iou, nan=-np.inf))
    xlabel = "Reference objects ordered by TAM3C2 assigned ST-IoU"
else:
    raise ValueError("Unknown reference_order_mode")

x = np.arange(n_reference_objects)
m_values, t_values = m3c2_iou[order], tam3c2_iou[order]
fig, ax = plt.subplots(
    figsize=(max(11, 0.42 * n_reference_objects), 6.2),
    dpi=130,
    constrained_layout=True,
)
for position, m_value, t_value in zip(x, m_values, t_values):
    if np.isfinite(m_value) and np.isfinite(t_value):
        ax.vlines(position, min(m_value, t_value), max(m_value, t_value),
                  color="0.68", lw=1.4, alpha=0.75, zorder=1)
ax.scatter(x[np.isfinite(m_values)], m_values[np.isfinite(m_values)],
           s=68, marker="o", color=METHOD_COLORS["M3C2"],
           edgecolor="white", lw=0.7, label="M3C2", zorder=3)
ax.scatter(x[np.isfinite(t_values)], t_values[np.isfinite(t_values)],
           s=53, marker="D", color=METHOD_COLORS["TAM3C2"],
           edgecolor="white", lw=0.7, label="TAM3C2", zorder=4)
ax.axhline(match_iou_threshold, color="black", ls="--", lw=1.1,
           label=f"Match threshold = {match_iou_threshold:.2f}")
ax.set(
    xlabel=xlabel,
    ylabel="Assigned spatiotemporal IoU",
    title="GT-centred one-to-one assignment comparison",
    ylim=(-0.03, 1.03),
)
ax.set_xticks(x, order)
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
plt.show()

both_finite=np.isfinite(m3c2_iou)&np.isfinite(tam3c2_iou)
tam3c2_higher=both_finite&(tam3c2_iou>m3c2_iou+IOU_EPS)
m3c2_higher=both_finite&(m3c2_iou>tam3c2_iou+IOU_EPS)
equal_iou=both_finite&np.isclose(
    m3c2_iou,tam3c2_iou,atol=IOU_EPS,rtol=0.0
)
not_comparable=~both_finite

comparison_groups={
    "Successful-match status":{
        "Only TAM3C2 successful":reference_indices[
            tam3c2_success&~m3c2_success
        ],
        "Only M3C2 successful":reference_indices[
            m3c2_success&~tam3c2_success
        ],
        "Both methods successful":reference_indices[
            m3c2_success&tam3c2_success
        ],
        "Neither method successful":reference_indices[
            ~m3c2_success&~tam3c2_success
        ],
    },
    "Assigned ST-IoU comparison":{
        "TAM3C2 has higher assigned ST-IoU":reference_indices[
            tam3c2_higher
        ],
        "M3C2 has higher assigned ST-IoU":reference_indices[
            m3c2_higher
        ],
        "Both methods have equal assigned ST-IoU":reference_indices[
            equal_iou
        ],
        "Cannot compare because one or both methods are unassigned":
            reference_indices[not_comparable],
    },
}

gt_one_to_one_summary_df=pd.DataFrame([
    {
        "summary_group":group,
        "result_type":label,
        "count":len(indices),
        "percentage_of_references":100*len(indices)/n_reference_objects,
        "reference_object_indices":list(map(int,indices)),
    }
    for group,categories in comparison_groups.items()
    for label,indices in categories.items()
])

display(gt_one_to_one_summary_df.style.format(
    {"percentage_of_references":"{:.1f}%"}
))


assigned IoU 为 0，但该 detection 实际存在更好候选 GT的对象：

In [ ]:
assigned=gt_assignment_df.loc[
    gt_assignment_df["detected_object_index"].notna()
].copy()

assigned["detected_object_index"]=(
    assigned["detected_object_index"].astype(int)
)

best_candidates=(
    pairwise_metrics_df
    .sort_values(
        ["method","detected_object_index","spatiotemporal_iou"],
        ascending=[True,True,False],
    )
    .groupby(
        ["method","detected_object_index"],
        as_index=False,
    )
    .first()
    .rename(columns={
        "reference_object_index":"best_reference_object_index",
        "spatial_iou":"best_spatial_iou",
        "temporal_iou":"best_temporal_iou",
        "spatiotemporal_iou":"best_spatiotemporal_iou",
    })
)

assignment_diagnostics_df=assigned.merge(
    best_candidates[[
        "method",
        "detected_object_index",
        "best_reference_object_index",
        "best_spatial_iou",
        "best_temporal_iou",
        "best_spatiotemporal_iou",
    ]],
    on=["method","detected_object_index"],
    how="left",
)

zero_assigned_but_better_exists_df=assignment_diagnostics_df.loc[
    (assignment_diagnostics_df["spatiotemporal_iou"]<=IOU_EPS)
    &(assignment_diagnostics_df["best_spatiotemporal_iou"]>IOU_EPS)
].copy()

display(
    zero_assigned_but_better_exists_df[[
        "method",
        "detected_object_index",
        "reference_object_index",
        "spatial_iou",
        "temporal_iou",
        "spatiotemporal_iou",
        "best_reference_object_index",
        "best_spatial_iou",
        "best_temporal_iou",
        "best_spatiotemporal_iou",
    ]]
)

### 5.1 Ranked quality of all Hungarian assignments

All object pairs selected by the Hungarian one-to-one assignment are retained, including pairs below the formal match threshold and pairs with zero ST-IoU. Within each method, assigned pairs are sorted independently from highest to lowest ST-IoU. Therefore, the x-axis is assignment rank rather than reference-object index, and the same rank does not represent the same GT object across methods.


In [ ]:
fig,ax=plt.subplots(figsize=(11,5.8),dpi=130,constrained_layout=True)
assignment_summary_rows=[]
total_counts={method:len(analyses[method].objects) for method in METHODS}

for method in METHODS:
    n_detected=total_counts[method]
    method_assignments=gt_assignment_df.loc[
        (gt_assignment_df["method"]==method)
        &gt_assignment_df["detected_object_index"].notna()
    ].copy()

    assigned_ious=method_assignments[
        "spatiotemporal_iou"
    ].fillna(0.0).to_numpy(float)

    assigned_indices=set(
        method_assignments["detected_object_index"].astype(int)
    )
    n_assigned=len(assigned_indices)
    n_unassigned=n_detected-n_assigned

    all_scores=np.concatenate([
        assigned_ious,
        np.zeros(n_unassigned,dtype=float),
    ])
    ranked_scores=np.sort(all_scores)[::-1]
    ranks=np.arange(1,n_detected+1)

    ax.plot(
        ranks,ranked_scores,
        marker="o",markersize=4,
        linewidth=1.5,
        color=METHOD_COLORS[method],
        label=f"{method} (total detected n={n_detected})",
        zorder=3,
    )

    # Clearly mark where each method ends.
    ax.axvline(
        n_detected,
        color=METHOD_COLORS[method],
        linestyle=":",
        linewidth=1.8,
        alpha=0.9,
        zorder=2,
    )
    ax.scatter(
        n_detected,ranked_scores[-1],
        marker="X",s=110,
        color=METHOD_COLORS[method],
        edgecolor="white",linewidth=0.8,
        zorder=5,
    )
    ax.annotate(
        f"{method} ends: n={n_detected}",
        xy=(n_detected,ranked_scores[-1]),
        xytext=(n_detected+1.2,0.055),
        rotation=90,
        ha="left",va="bottom",
        color=METHOD_COLORS[method],
        fontsize=10,fontweight="bold",
    )

    n_successful=int(np.sum(assigned_ious>=match_iou_threshold))
    n_assigned_zero=int(np.sum(assigned_ious<=IOU_EPS))

    assignment_summary_rows.append({
        "method":method,
        "total_detected_objects":n_detected,
        "hungarian_assigned_objects":n_assigned,
        "unassigned_detected_objects":n_unassigned,
        "successful_assignments":n_successful,
        "assigned_pairs_with_zero_ST_IoU":n_assigned_zero,
        "total_objects_plotted_at_zero":n_assigned_zero+n_unassigned,
    })

ax.axhline(
    match_iou_threshold,
    color="black",linestyle="--",linewidth=1.1,
    label=f"Match threshold={match_iou_threshold:.2f}",
)

ax.set(
    xlabel="Detected-object rank by assigned spatiotemporal IoU",
    ylabel="Assigned spatiotemporal IoU",
    title="Ranked detected-object quality after Hungarian one-to-one assignment",
    xlim=(0,max(total_counts.values())+10),
    ylim=(-0.02,1.03),
)

ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(alpha=0.25)
ax.legend(frameon=False)
plt.show()

assignment_plot_summary_df=pd.DataFrame(assignment_summary_rows)
display(assignment_plot_summary_df)

## 6. Non-zero overlap relationships

List every detected-reference pair with positive joint ST-IoU in both directions. These tables reveal which detections contribute to each GT and which GT objects are touched by each detection; official one-to-one matches are flagged but the full overlap graph is retained.


In [ ]:
overlap_pairs_df = pairwise_metrics_df.loc[
    pairwise_metrics_df["spatiotemporal_iou"].fillna(0) > IOU_EPS
].copy()

official_keys = matched_objects_df[
    ["method", "detected_object_index", "reference_object_index"]
].drop_duplicates().assign(official_one_to_one_match=True)
overlap_pairs_df = overlap_pairs_df.merge(
    official_keys,
    on=["method", "detected_object_index", "reference_object_index"],
    how="left",
)
overlap_pairs_df["official_one_to_one_match"] = (
    overlap_pairs_df["official_one_to_one_match"].fillna(False).astype(bool)
)

def compact_relationships(frame, group_column, related_column):
    output_column = related_column.replace("_index", "_indices")
    return (
        frame.sort_values(
            ["method", group_column, "spatiotemporal_iou"],
            ascending=[True, True, False],
        )
        .groupby(["method", group_column], as_index=False)
        .agg(**{
            output_column: (
                related_column, lambda values: list(map(int, values))
            ),
            "spatial_ious": (
                "spatial_iou", lambda values: list(np.round(values, 4))
            ),
            "temporal_ious": (
                "temporal_iou", lambda values: list(np.round(values, 4))
            ),
            "spatiotemporal_ious": (
                "spatiotemporal_iou", lambda values: list(np.round(values, 4))
            ),
        })
    )

reference_to_detected_df = compact_relationships(
    overlap_pairs_df, "reference_object_index", "detected_object_index"
)
detected_to_reference_df = compact_relationships(
    overlap_pairs_df, "detected_object_index", "reference_object_index"
)
display(reference_to_detected_df)
display(detected_to_reference_df)

diagnostic_output_dir = os.path.join(archive_folder, "object_overlap_diagnostics")
os.makedirs(diagnostic_output_dir, exist_ok=True)
overlap_pairs_df.to_csv(
    os.path.join(diagnostic_output_dir, "all_nonzero_overlap_pairs.csv"),
    index=False,
)
reference_to_detected_df.to_csv(
    os.path.join(diagnostic_output_dir, "reference_to_detected_index_lists.csv"),
    index=False,
)
detected_to_reference_df.to_csv(
    os.path.join(diagnostic_output_dir, "detected_to_reference_index_lists.csv"),
    index=False,
)
print(f"Saved overlap diagnostics to: {diagnostic_output_dir}")


## 7. Detected-object overlap topology

Treat positive ST-IoU pairs as edges in a detected-reference graph. A clean one-to-one object has degree one on both sides; multiple detections connected to one GT indicate fragmentation, while one detection connected to multiple GT objects indicates merging. Categories are mutually exclusive, so each stacked bar sums exactly to the method's detected-object total.


In [ ]:
CATEGORY_ORDER = [
    "Clean one-to-one, IoU >= 0.8",
    "Clean one-to-one, IoU < 0.8",
    "Fragmentation only",
    "Merging only",
    "Fragmentation + merging",
    "Zero joint ST overlap",
]
CATEGORY_COLORS = {
    "Clean one-to-one, IoU >= 0.8": "#54A24B",
    "Clean one-to-one, IoU < 0.8": "#9EC5AB",
    "Fragmentation only": "#B279A2",
    "Merging only": "#FF9DA6",
    "Fragmentation + merging": "#ECA82C",
    "Zero joint ST overlap": "#E45756",
}

def component_pattern(spatial_iou, temporal_iou):
    spatial = "high" if spatial_iou >= match_iou_threshold else "low"
    temporal = "high" if temporal_iou >= match_iou_threshold else "low"
    return f"Spatial {spatial}; temporal {temporal}"


classification_rows, relation_rows, zero_rows = [], [], []
for method in METHODS:
    pairs = pairwise_metrics_df.query("method == @method").copy()
    edges = pairs.loc[
        pairs["spatiotemporal_iou"].fillna(0) > topology_iou_threshold
    ]
    refs_by_det = edges.groupby("detected_object_index")[
        "reference_object_index"
    ].agg(lambda values: set(map(int, values))).to_dict()
    dets_by_ref = edges.groupby("reference_object_index")[
        "detected_object_index"
    ].agg(lambda values: set(map(int, values))).to_dict()

    for edge in edges.itertuples(index=False):
        detected_idx = int(edge.detected_object_index)
        reference_idx = int(edge.reference_object_index)
        fragment = len(dets_by_ref[reference_idx]) > 1
        merged = len(refs_by_det[detected_idx]) > 1
        relation_rows.append({
            "method": method,
            "detected_object_index": detected_idx,
            "reference_object_index": reference_idx,
            "spatial_iou": float(edge.spatial_iou),
            "temporal_iou": float(edge.temporal_iou),
            "spatiotemporal_iou": float(edge.spatiotemporal_iou),
            "fragment_relation": fragment,
            "merged_relation": merged,
            "component_pattern": component_pattern(
                float(edge.spatial_iou), float(edge.temporal_iou)
            ),
            "limiting_dimension": (
                "Spatial" if edge.spatial_iou < edge.temporal_iou
                else "Temporal" if edge.temporal_iou < edge.spatial_iou
                else "Equal"
            ),
        })

    for detected_idx in range(len(analyses[method].objects)):
        detected_pairs = pairs.query("detected_object_index == @detected_idx")
        best_iou = detected_pairs["spatiotemporal_iou"].fillna(-np.inf)
        best = detected_pairs.loc[best_iou.idxmax()]
        connected_refs = refs_by_det.get(detected_idx, set())
        fragment = any(len(dets_by_ref[ref_idx]) > 1 for ref_idx in connected_refs)
        merged = len(connected_refs) > 1

        if not connected_refs:
            category = "Zero joint ST overlap"
            max_spatial = float(detected_pairs["spatial_iou"].max())
            max_temporal = float(detected_pairs["temporal_iou"].max())
            if max_spatial <= IOU_EPS and max_temporal <= IOU_EPS:
                reason, diagnostic_ref, diagnostic_value = (
                    "No spatial or temporal overlap", np.nan, 0.0
                )
                diagnostic_spatial = diagnostic_temporal = 0.0
            elif max_spatial >= max_temporal:
                row = detected_pairs.loc[detected_pairs["spatial_iou"].idxmax()]
                reason, diagnostic_value = "Spatial overlap only", float(row.spatial_iou)
                diagnostic_ref = int(row.reference_object_index)
                diagnostic_spatial, diagnostic_temporal = float(row.spatial_iou), float(row.temporal_iou)
            else:
                row = detected_pairs.loc[detected_pairs["temporal_iou"].idxmax()]
                reason, diagnostic_value = "Temporal overlap only", float(row.temporal_iou)
                diagnostic_ref = int(row.reference_object_index)
                diagnostic_spatial, diagnostic_temporal = float(row.spatial_iou), float(row.temporal_iou)
            zero_rows.append({
                "method": method,
                "detected_object_index": detected_idx,
                "zero_st_reason": reason,
                "dominant_reference_object_index": diagnostic_ref,
                "spatial_iou": diagnostic_spatial,
                "temporal_iou": diagnostic_temporal,
                "diagnostic_overlap_value": diagnostic_value,
            })
        elif fragment and merged:
            category = "Fragmentation + merging"
        elif fragment:
            category = "Fragmentation only"
        elif merged:
            category = "Merging only"
        elif best.spatiotemporal_iou >= match_iou_threshold:
            category = "Clean one-to-one, IoU >= 0.8"
        else:
            category = "Clean one-to-one, IoU < 0.8"

        classification_rows.append({
            "method": method,
            "detected_object_index": detected_idx,
            "primary_category": category,
            "best_reference_object_index": int(best.reference_object_index),
            "best_spatiotemporal_iou": float(best.spatiotemporal_iou),
            "is_fragment_candidate": fragment,
            "is_merged_candidate": merged,
        })

detected_topology_classification_df = pd.DataFrame(classification_rows)
topology_relation_df = pd.DataFrame(relation_rows)
zero_st_overlap_detail_df = pd.DataFrame(zero_rows)

classification_counts_df = (
    detected_topology_classification_df
    .groupby(["method", "primary_category"]).size().unstack(fill_value=0)
    .reindex(index=METHODS, columns=CATEGORY_ORDER, fill_value=0)
)
classification_counts_df["Classified total"] = classification_counts_df.sum(axis=1)
classification_counts_df["Detected-object total"] = [
    len(analyses[method].objects) for method in METHODS
]
if not (
    classification_counts_df["Classified total"]
    == classification_counts_df["Detected-object total"]
).all():
    raise AssertionError("Topology categories do not sum to detected-object totals")
display(classification_counts_df)

CATEGORY_LABELS = {
    "Clean one-to-one, IoU >= 0.8": (
        "Clean 1-to-1\nIoU ≥ 0.8"
    ),
    "Clean one-to-one, IoU < 0.8": (
        "Clean 1-to-1\nIoU < 0.8"
    ),
    "Fragmentation only": (
        "Fragmentation"
    ),
    "Merging only": (
        "Merging"
    ),
    "Fragmentation + merging": (
        "Fragmentation\n+ merging"
    ),
    "Zero joint ST overlap": (
        "Zero joint\nST overlap"
    ),
}
plot_categories = CATEGORY_ORDER
x_positions = np.arange(
    len(plot_categories)
)
bar_width = 0.38
fig, ax = plt.subplots(
    figsize=(12, 6.5),
    dpi=130,
    constrained_layout=True,
)

for method_index, method in enumerate(METHODS):
    counts = (
        classification_counts_df.loc[
            method,
            plot_categories,
        ]
        .to_numpy(dtype=float)
    )
    detected_total = len(analyses[method].objects )
    percentages = (
        100.0 * counts / detected_total
        if detected_total > 0
        else np.zeros_like(counts)
    )
    offset = (method_index - 0.5) * bar_width
    bars = ax.bar(
        x_positions + offset,
        counts,
        width=bar_width,
        color=METHOD_COLORS[method],
        edgecolor="white",
        linewidth=0.8,
        label=(
            f"{method} "
            f"(total = {detected_total})"
        ),
        zorder=3,
    )
    for bar, count, percentage in zip(
        bars,
        counts,
        percentages,
    ):
        if count <= 0:
            continue

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            (
                f"{int(count)}\n"
                f"({percentage:.1f}%)"
            ),
            ha="center",
            va="bottom",
            fontsize=9,
        )
ax.set_xticks(x_positions)
ax.set_xticklabels( [
        CATEGORY_LABELS[category]
        for category in plot_categories
    ]
)
ax.set_xlabel("Detected-object overlap category")
ax.set_ylabel("Number of detected objects")
ax.set_title("Detected-object overlap topology by method")
ax.grid(
    axis="y",
    alpha=0.25,
    zorder=0,
)
ax.set_axisbelow(True)
# Add enough space above the tallest bar for labels.
maximum_count = (
    classification_counts_df.loc[
        list(METHODS),
        plot_categories,
    ]
    .to_numpy(dtype=float)
    .max()
)
ax.set_ylim(0, maximum_count * 1.20
    if maximum_count > 0
    else 1,)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False,)
plt.show()

category_index_lists_df = (
    detected_topology_classification_df
    .groupby(["method", "primary_category"])["detected_object_index"]
    .agg(lambda values: sorted(map(int, values)))
    .reset_index(name="detected_object_indices")
)
category_index_lists_df["count"] = category_index_lists_df[
    "detected_object_indices"
].apply(len)
display(category_index_lists_df)


### 7.1 Fragmentation and merging diagnostics

Summaries count affected detections, references, and pairwise relations. Spatial/temporal high-low patterns use the formal 0.8 threshold separately on each component; the limiting dimension indicates whether spatial or temporal agreement is weaker. Detailed pair tables are saved rather than fully rendered to keep the notebook compact.


n fragment detection, n_references, n_relations:参与 fragmentation 的不同 detected objects, reference objects数量;detected–reference overlap pairs;  
mean iou:所有被筛选 relation 的平均值 (每个relation pair);  
fragment和merge在两个diagnose 中会重复，所以和柱状图不一致。  
Spatial high; temporal high 表格中的数值是 relation 数量，不是 detected object 数量。

In [ ]:
def relation_diagnostics(frame, flag, detection_label, reference_label):
    detail = frame.loc[frame[flag]].copy()
    if detail.empty:
        return detail, pd.DataFrame(index=METHODS), pd.DataFrame(index=METHODS)
    summary = (
        detail.groupby("method")
        .agg(
            **{
                detection_label: ("detected_object_index", "nunique"),
                reference_label: ("reference_object_index", "nunique"),
                "n_relations": ("spatiotemporal_iou", "size"),
                "mean_spatial_iou": ("spatial_iou", "mean"),
                "mean_temporal_iou": ("temporal_iou", "mean"),
                "mean_spatiotemporal_iou": ("spatiotemporal_iou", "mean"),
            }
        )
        .reindex(METHODS).fillna(0)
    )
    patterns = (
        detail.groupby(["method", "component_pattern"]).size()
        .unstack(fill_value=0).reindex(METHODS, fill_value=0)
    )
    return detail, summary, patterns


fragment_detail_df, fragment_summary_df, fragment_patterns_df = relation_diagnostics(
    topology_relation_df, "fragment_relation",
    "n_fragment_detections", "n_fragmented_references",
)
merged_detail_df, merged_summary_df, merged_patterns_df = relation_diagnostics(
    topology_relation_df, "merged_relation",
    "n_merged_detections", "n_affected_references",
)

print("Fragmentation summary")
display(fragment_summary_df)
display(fragment_patterns_df)
print("Merging summary")
display(merged_summary_df)
display(merged_patterns_df)

fragment_detail_df.to_csv(
    os.path.join(diagnostic_output_dir, "fragmentation_relations.csv"), index=False
)
merged_detail_df.to_csv(
    os.path.join(diagnostic_output_dir, "merging_relations.csv"), index=False
)


In [ ]:
# Grouped-union evaluation for fragmentation and merging.

fragment_union_rows=[]
fragment_relations=topology_relation_df.loc[
    topology_relation_df["fragment_relation"]
]

for (method,reference_idx),relations in fragment_relations.groupby(
    ["method","reference_object_index"]
):
    reference_idx=int(reference_idx)
    detected_indices=sorted(
        set(relations["detected_object_index"].astype(int))
    )
    best_relation=relations.loc[
        relations["spatiotemporal_iou"].idxmax()
    ]
    _,det_time_days=timestamp_info[method]
    _,ref_time_days=timestamp_info["GT"]

    union_metrics=_global_support_metrics(
        [analyses[method].objects[idx] for idx in detected_indices],
        [analyses["GT"].objects[reference_idx]],
        det_time_days,
        ref_time_days,
    )

    best_iou=float(best_relation["spatiotemporal_iou"])
    union_iou=float(union_metrics["global_iou"])

    fragment_union_rows.append({
        "method":method,
        "reference_object_index":reference_idx,
        "n_fragments":len(detected_indices),
        "detected_object_indices":detected_indices,
        "best_fragment_index":int(
            best_relation["detected_object_index"]
        ),
        "best_individual_fragment_iou":best_iou,
        "fragment_union_iou":union_iou,
        "union_iou_gain":union_iou-best_iou,
        "fragment_union_precision":union_metrics["global_precision"],
        "fragment_union_recall":union_metrics["global_recall"],
        "union_improves_iou":union_iou>best_iou+IOU_EPS,
        "rescued_at_threshold":(
            best_iou<match_iou_threshold
            and union_iou>=match_iou_threshold
        ),
    })

fragment_grouped_union_df=pd.DataFrame(fragment_union_rows).sort_values(
    ["method","union_iou_gain"],
    ascending=[True,False],
).reset_index(drop=True)

fragment_grouped_summary_df=(
    fragment_grouped_union_df.groupby("method")
    .agg(
        n_fragmented_references=("reference_object_index","nunique"),
        mean_n_fragments=("n_fragments","mean"),
        mean_best_individual_iou=(
            "best_individual_fragment_iou","mean"
        ),
        mean_fragment_union_iou=("fragment_union_iou","mean"),
        mean_union_iou_gain=("union_iou_gain","mean"),
        n_union_improved=("union_improves_iou","sum"),
        n_rescued_at_threshold=("rescued_at_threshold","sum"),
    )
    .reindex(METHODS)
    .reset_index()
)

print("Fragment grouped-union summary")
display(fragment_grouped_summary_df.round({
    "mean_n_fragments":2,
    "mean_best_individual_iou":3,
    "mean_fragment_union_iou":3,
    "mean_union_iou_gain":3,
}))

print("Fragment grouped-union details")
display(fragment_grouped_union_df.round({
    "best_individual_fragment_iou":3,
    "fragment_union_iou":3,
    "union_iou_gain":3,
    "fragment_union_precision":3,
    "fragment_union_recall":3,
}))


merged_union_rows=[]
merged_relations=topology_relation_df.loc[
    topology_relation_df["merged_relation"]
]

for (method,detected_idx),relations in merged_relations.groupby(
    ["method","detected_object_index"]
):
    detected_idx=int(detected_idx)
    reference_indices_for_detection=sorted(
        set(relations["reference_object_index"].astype(int))
    )
    best_relation=relations.loc[
        relations["spatiotemporal_iou"].idxmax()
    ]
    _,det_time_days=timestamp_info[method]
    _,ref_time_days=timestamp_info["GT"]

    union_metrics=_global_support_metrics(
        [analyses[method].objects[detected_idx]],
        [
            analyses["GT"].objects[idx]
            for idx in reference_indices_for_detection
        ],
        det_time_days,
        ref_time_days,
    )

    best_iou=float(best_relation["spatiotemporal_iou"])
    union_iou=float(union_metrics["global_iou"])

    merged_union_rows.append({
        "method":method,
        "detected_object_index":detected_idx,
        "n_connected_references":len(
            reference_indices_for_detection
        ),
        "reference_object_indices":reference_indices_for_detection,
        "best_reference_index":int(
            best_relation["reference_object_index"]
        ),
        "best_individual_reference_iou":best_iou,
        "gt_union_iou":union_iou,
        "union_iou_gain":union_iou-best_iou,
        "gt_union_precision":union_metrics["global_precision"],
        "gt_union_recall":union_metrics["global_recall"],
        "union_improves_iou":union_iou>best_iou+IOU_EPS,
        "rescued_at_threshold":(
            best_iou<match_iou_threshold
            and union_iou>=match_iou_threshold
        ),
    })

merged_gt_union_df=pd.DataFrame(merged_union_rows).sort_values(
    ["method","union_iou_gain"],
    ascending=[True,False],
).reset_index(drop=True)

merged_gt_union_summary_df=(
    merged_gt_union_df.groupby("method")
    .agg(
        n_merged_detections=("detected_object_index","nunique"),
        mean_n_connected_references=(
            "n_connected_references","mean"
        ),
        mean_best_individual_iou=(
            "best_individual_reference_iou","mean"
        ),
        mean_gt_union_iou=("gt_union_iou","mean"),
        mean_union_iou_gain=("union_iou_gain","mean"),
        n_union_improved=("union_improves_iou","sum"),
        n_rescued_at_threshold=("rescued_at_threshold","sum"),
    )
    .reindex(METHODS)
    .reset_index()
)

print("Merged-detection versus GT-union summary")
display(merged_gt_union_summary_df.round({
    "mean_n_connected_references":2,
    "mean_best_individual_iou":3,
    "mean_gt_union_iou":3,
    "mean_union_iou_gain":3,
}))

print("Merged-detection versus GT-union details")
display(merged_gt_union_df.round({
    "best_individual_reference_iou":3,
    "gt_union_iou":3,
    "union_iou_gain":3,
    "gt_union_precision":3,
    "gt_union_recall":3,
}))

### 7.2 Zero joint ST-overlap diagnostics

Every zero-ST detection is assigned to one of three mutually exclusive primary reasons using its strongest partial relation: spatial overlap only, temporal overlap only, or neither. The three counts therefore sum exactly to the zero-ST total; the detailed table records the selected reference and component IoU.


In [ ]:
ZERO_REASON_ORDER = [
    "Spatial overlap only",
    "Temporal overlap only",
    "No spatial or temporal overlap",
]
zero_st_overlap_counts_df = (
    zero_st_overlap_detail_df.groupby(["method", "zero_st_reason"]).size()
    .unstack(fill_value=0)
    .reindex(index=METHODS, columns=ZERO_REASON_ORDER, fill_value=0)
)
zero_st_overlap_counts_df["Zero-ST total"] = zero_st_overlap_counts_df.sum(axis=1)
expected_zero = (
    detected_topology_classification_df.query(
        "primary_category == 'Zero joint ST overlap'"
    ).groupby("method").size().reindex(METHODS, fill_value=0)
)
if not (zero_st_overlap_counts_df["Zero-ST total"] == expected_zero).all():
    raise AssertionError("Zero-ST reason counts do not match the topology classification")
display(zero_st_overlap_counts_df)

zero_st_overlap_lists_df = (
    zero_st_overlap_detail_df
    .groupby(["method", "zero_st_reason"])
    .agg(
        detected_object_indices=(
            "detected_object_index", lambda values: list(map(int, values))
        ),
        dominant_reference_indices=(
            "dominant_reference_object_index",
            lambda values: [int(v) if np.isfinite(v) else None for v in values],
        ),
        spatial_iou_values=(
            "spatial_iou", lambda values: list(np.round(values, 4))
        ),
        temporal_iou_values=(
            "temporal_iou", lambda values: list(np.round(values, 4))
        ),
    )
    .reset_index()
)
zero_st_overlap_lists_df["count"] = zero_st_overlap_lists_df[
    "detected_object_indices"
].apply(len)
display(zero_st_overlap_lists_df)
zero_st_overlap_detail_df.to_csv(
    os.path.join(diagnostic_output_dir, "zero_st_overlap_objects.csv"), index=False
)


## 9. Individual-object investigation

This optional final section inspects selected objects after the aggregate evaluation is complete. It first maps all object outlines, then chooses one GT object and finds each method's best-overlap detection for diagnostic visualization. These best-overlap selections are reference-centred and are not replacements for the formal Hungarian matches above.


In [ ]:
def _objects_for_analysis(st_analysis):
    if st_analysis.objects is None:
        raise RuntimeError(f"{st_analysis.filename} has no stored 4D-OBC objects")
    return list(st_analysis.objects)


def _object_indices(obj):
    return np.asarray(obj.indices, dtype=int)


def _object_xy(st_analysis, obj):
    return np.asarray(st_analysis.corepoints.cloud)[_object_indices(obj), :2]
# Shared plotting helpers.

# Shared publication style. Font sizes are specified in points, while the
# panel geometry is specified in inches. The standalone and combined changemap
# axes therefore have exactly the same physical size.
OBJECT_FIG_DPI = 120  # notebook display only; use 300 dpi for PNG export

OBJECT_FONT_SIZE = 14
OBJECT_TITLE_SIZE = 14
OBJECT_TICK_SIZE = 14
OBJECT_LEGEND_SIZE = 14
OBJECT_ANNOTATION_SIZE = 14
OBJECT_COLORBAR_SIZE = 12
OBJECT_COLORBAR_TICK_SIZE = 12

# Fixed physical geometry for every changemap panel.
OBJECT_PANEL_HEIGHT = 4.80
OBJECT_MAP_WIDTH = 4.80
OBJECT_TIME_SERIES_WIDTH = 9.60
OBJECT_LEFT_MARGIN = 0.95
OBJECT_RIGHT_MARGIN = 0.20
OBJECT_BOTTOM_MARGIN = 0.90
OBJECT_TOP_MARGIN = 0.50
OBJECT_PANEL_GAP = 0.70
OBJECT_COLORBAR_GAP = 0.16
OBJECT_COLORBAR_WIDTH = 0.25

OBJECT_COMMON_FIGURE_HEIGHT = (
    OBJECT_BOTTOM_MARGIN + OBJECT_PANEL_HEIGHT + OBJECT_TOP_MARGIN
)
OBJECT_SINGLE_CHANGEMAP_FIGSIZE = (
    OBJECT_LEFT_MARGIN
    + OBJECT_MAP_WIDTH
    + OBJECT_COLORBAR_GAP
    + OBJECT_COLORBAR_WIDTH
    + OBJECT_RIGHT_MARGIN,
    OBJECT_COMMON_FIGURE_HEIGHT,
)
OBJECT_TIME_SERIES_MAP_FIGSIZE = (
    OBJECT_LEFT_MARGIN
    + OBJECT_TIME_SERIES_WIDTH
    + OBJECT_PANEL_GAP
    + OBJECT_MAP_WIDTH
    + OBJECT_COLORBAR_GAP
    + OBJECT_COLORBAR_WIDTH
    + OBJECT_RIGHT_MARGIN,
    OBJECT_COMMON_FIGURE_HEIGHT,
)
OBJECT_TEMPORAL_FIGSIZE = (OBJECT_TIME_SERIES_MAP_FIGSIZE[0], 5.4)


def _axes_box(fig_width, fig_height, left, bottom, width, height):
    """Convert an axes rectangle from inches to figure fractions."""
    return [
        left / fig_width,
        bottom / fig_height,
        width / fig_width,
        height / fig_height,
    ]


def _new_single_changemap_figure():
    fig_width, fig_height = OBJECT_SINGLE_CHANGEMAP_FIGSIZE
    fig = plt.figure(
        figsize=OBJECT_SINGLE_CHANGEMAP_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
    )
    ax_map = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            OBJECT_LEFT_MARGIN,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_MAP_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    colorbar_left = (
        OBJECT_LEFT_MARGIN + OBJECT_MAP_WIDTH + OBJECT_COLORBAR_GAP
    )
    ax_colorbar = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            colorbar_left,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_COLORBAR_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    return fig, ax_map, ax_colorbar


def _new_time_series_changemap_figure():
    fig_width, fig_height = OBJECT_TIME_SERIES_MAP_FIGSIZE
    fig = plt.figure(
        figsize=OBJECT_TIME_SERIES_MAP_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
    )
    ax_time_series = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            OBJECT_LEFT_MARGIN,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_TIME_SERIES_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    map_left = (
        OBJECT_LEFT_MARGIN + OBJECT_TIME_SERIES_WIDTH + OBJECT_PANEL_GAP
    )
    ax_map = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            map_left,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_MAP_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    colorbar_left = map_left + OBJECT_MAP_WIDTH + OBJECT_COLORBAR_GAP
    ax_colorbar = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            colorbar_left,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_COLORBAR_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    return fig, ax_time_series, ax_map, ax_colorbar

def _apply_object_plot_font_sizes(fig):
    for ax in fig.axes:
        is_colorbar = bool(getattr(ax, "_is_object_colorbar", False))

        ax.title.set_fontsize(OBJECT_TITLE_SIZE if not is_colorbar else OBJECT_COLORBAR_SIZE)
        ax.xaxis.label.set_fontsize(OBJECT_FONT_SIZE if not is_colorbar else OBJECT_COLORBAR_SIZE)
        ax.yaxis.label.set_fontsize(OBJECT_FONT_SIZE if not is_colorbar else OBJECT_COLORBAR_SIZE)
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=OBJECT_COLORBAR_TICK_SIZE if is_colorbar else OBJECT_TICK_SIZE,
        )
        ax.tick_params(
            axis="both",
            which="minor",
            labelsize=OBJECT_COLORBAR_TICK_SIZE if is_colorbar else OBJECT_TICK_SIZE,
        )

        for text in ax.texts:
            text.set_fontsize(OBJECT_ANNOTATION_SIZE)

        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(OBJECT_LEGEND_SIZE)
            legend.get_title().set_fontsize(OBJECT_LEGEND_SIZE)

def _draw_object_extent(
    ax,
    xy,
    color,
    label,
    linewidth=2.2,
    point_size=10,
    point_alpha=0.55,
    fill_alpha=0.05,
):
    xy = np.asarray(xy, dtype=float)
    if len(xy) == 0:
        return

    xy_unique = np.unique(xy, axis=0)

    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=point_size,
        color=color,
        alpha=point_alpha,
        linewidths=0,
        zorder=4,
    )

    if len(xy_unique) >= 3:
        try:
            hull = ConvexHull(xy_unique)
            hull_xy = xy_unique[hull.vertices]

            ax.add_patch(
                Polygon(
                    hull_xy,
                    closed=True,
                    facecolor=color,
                    edgecolor=color,
                    linewidth=linewidth,
                    alpha=fill_alpha,
                    zorder=5,
                )
            )
            ax.plot(
                np.r_[hull_xy[:, 0], hull_xy[0, 0]],
                np.r_[hull_xy[:, 1], hull_xy[0, 1]],
                color=color,
                linewidth=linewidth,
                label=label,
                zorder=6,
            )
            return
        except QhullError:
            pass

    if len(xy_unique) >= 2:
        order = np.lexsort((xy_unique[:, 1], xy_unique[:, 0]))
        xy_line = xy_unique[order]
        ax.plot(
            xy_line[:, 0],
            xy_line[:, 1],
            color=color,
            linewidth=linewidth,
            label=label,
            zorder=6,
        )
    else:
        ax.scatter(
            xy_unique[:, 0],
            xy_unique[:, 1],
            s=40,
            color=color,
            label=label,
            zorder=6,
        )

def _raw_changemap(record):
    name = record["name"]
    st_analysis = analyses[name]

    cloud = np.asarray(st_analysis.corepoints.cloud)
    distances = np.asarray(st_analysis.distances, dtype=float)

    start_epoch = int(record["interval"]["start_epoch"])
    end_epoch = int(record["interval"]["end_epoch"])
    magnitudes = distances[:, end_epoch] - distances[:, start_epoch]

    return cloud, magnitudes

def _scene_limits(ax, cloud, pad_ratio=0.02):
    xy = np.asarray(cloud[:, :2], dtype=float)
    x_min, y_min = np.nanmin(xy, axis=0)
    x_max, y_max = np.nanmax(xy, axis=0)

    span = max(x_max - x_min, y_max - y_min, 1.0)
    pad = pad_ratio * span

    ax.set_xlim(x_min - pad, x_max + pad)
    ax.set_ylim(y_min - pad, y_max + pad)

def _extent_limits(records, pad_ratio=0.35, min_span=2.0):
    object_xy = [np.asarray(record["xy"], dtype=float) for record in records if len(record["xy"]) > 0]
    if not object_xy:
        return (-0.5 * min_span, 0.5 * min_span, -0.5 * min_span, 0.5 * min_span)

    xy = np.vstack(object_xy)
    x_min, y_min = np.nanmin(xy, axis=0)
    x_max, y_max = np.nanmax(xy, axis=0)

    x_center = 0.5 * (x_min + x_max)
    y_center = 0.5 * (y_min + y_max)

    span = max(x_max - x_min, y_max - y_min, min_span)
    half_span = 0.5 * span * (1.0 + 2.0 * pad_ratio)

    return (
        x_center - half_span,
        x_center + half_span,
        y_center - half_span,
        y_center + half_span,
    )

def _resolve_crange(magnitudes, crange):
    finite = np.asarray(magnitudes, dtype=float)
    finite = finite[np.isfinite(finite)]

    if crange is not None:
        resolved = float(crange)
        return resolved if resolved > 0 else 1.0

    if len(finite) == 0:
        return 1.0

    auto_crange = float(np.nanmax(np.abs(finite)))
    return auto_crange if auto_crange > 0 else 1.0

def _plot_spatial_panel(
    ax,
    records,
    background_record=None,
    title="",
    crange=None,
    zoom=False,
    add_colorbar=True,
    show_legend=True,
    legend_loc="upper right",
    legend_bbox_to_anchor=None,
    colorbar_ax=None,
):
    if background_record is None:
        background_record = next(
            (record for record in records if record["name"] == "GT"),
            records[0],
        )

    cloud, magnitudes = _raw_changemap(background_record)
    resolved_crange = _resolve_crange(magnitudes, crange)

    scatter = ax.scatter(
        cloud[:, 0],
        cloud[:, 1],
        c=magnitudes,
        cmap="seismic_r",
        vmin=-resolved_crange,
        vmax=resolved_crange,
        s=1.0,
        linewidths=0,
        alpha=0.85,
        zorder=1,
    )

    for record in records:
        _draw_object_extent(
            ax,
            record["xy"],
            color=record["color"],
            label=f"{record['name']} object {record['object_id']}",
            linewidth=2.6 if record["name"] == "GT" else 2.2,
            point_size=13 if record["name"] == "GT" else 10,
        )

    if zoom:
        x_min, x_max, y_min, y_max = _extent_limits(records)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
    else:
        _scene_limits(ax, cloud)

    ax.set_title(title)
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)

    if show_legend:
        ax.legend(
            loc=legend_loc,
            bbox_to_anchor=legend_bbox_to_anchor,
            framealpha=0.9,
            borderaxespad=0.2,
        )

    if add_colorbar:
        if colorbar_ax is None:
            cbar = ax.figure.colorbar(
                scatter,
                ax=ax,
                format="%.2f",
                fraction=0.046,
                pad=0.02,
            )
        else:
            cbar = ax.figure.colorbar(
                scatter,
                cax=colorbar_ax,
                format="%.2f",
            )
        cbar.set_label(f"Change magnitude [m], scale +/-{resolved_crange:.2f}")
        cbar.ax._is_object_colorbar = True

    return scatter


### 9.1 All-object spatial overview

Plot every stored object outline on the common core-point map. This is a qualitative check for differences in object count, location, fragmentation, and merging before selecting a single reference object.


In [ ]:
def plot_all_object_delineations(method_names=("GT", "M3C2", "TAM3C2")):
    fig, axes = plt.subplots(
        1, len(method_names), figsize=(6.2 * len(method_names), 6),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    cmap = plt.colormaps.get_cmap("tab20")
    for ax, name in zip(axes, method_names):
        st_analysis = analyses[name]
        cloud = np.asarray(st_analysis.corepoints.cloud)
        objects = _objects_for_analysis(st_analysis)
        ax.scatter(cloud[:, 0], cloud[:, 1], s=0.3, color="0.88", linewidths=0)
        for object_id, obj in enumerate(objects):
            xy = _object_xy(st_analysis, obj)
            _draw_object_extent(
                ax, xy, color=cmap(object_id % 20), label=None,
                linewidth=1.2, point_size=4, point_alpha=0.2,
            )
            if len(xy):
                centroid = np.mean(xy, axis=0)
                ax.text(centroid[0], centroid[1], str(object_id), fontsize=8,
                        ha="center", va="center",
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=1))
        ax.set(
            title=f"{name}: all 4D-OBCs (n={len(objects)})",
            xlabel="X [m]", ylabel="Y [m]",
        )
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)
    plt.show()


plot_all_object_delineations()


### 9.2 Selected-reference match summary

Set `reference_object_id`, independently find the maximum-ST-IoU M3C2 and TAM3C2 detections, and report their spatial, temporal, joint, duration, and change-volume agreement with that GT object.


In [ ]:
reference_object_id = 10

In [ ]:
# Select one GT/reference object and find its best M3C2/TAM3C2 matches.
use_smoothed_time_series = False
OBJECT_METHODS = ("GT", "M3C2", "TAM3C2")
OBJECT_COLORS = {"GT": "black", "M3C2": "tab:blue", "TAM3C2": "tab:orange"}

print("Object diagnostics use raw st_analysis.distances, not smoothed_distances.")


def _timestamp_info_for(name):
    if "timestamp_info" in globals() and name in timestamp_info:
        return timestamp_info[name]
    return _timestamps_for_analysis(analyses[name])


def _analysis_object(name, object_id):
    return _objects_for_analysis(analyses[name])[object_id]


def _single_object_change_summary(name, object_id):
    _, time_days = _timestamp_info_for(name)
    return _mean_time_integrated_change(
        _analysis_object(name, object_id),
        np.asarray(analyses[name].distances, dtype=float),
        time_days,
    )


def _safe_percent_ratio(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0:
        return np.nan
    return 100.0 * numerator / denominator


def _best_detected_match_for_reference(reference_object_id, method):
    gt_objects = _objects_for_analysis(analyses["GT"])
    detected_objects = _objects_for_analysis(analyses[method])

    if not 0 <= reference_object_id < len(gt_objects):
        raise IndexError(
            f"reference_object_id={reference_object_id} outside GT object range "
            f"[0, {len(gt_objects) - 1}]"
        )

    _, ref_time_days = _timestamp_info_for("GT")
    _, det_time_days = _timestamp_info_for(method)

    rows = []
    for detected_object_id, det_obj in enumerate(detected_objects):
        row = _pairwise_object_metrics(
            det_obj,
            gt_objects[reference_object_id],
            det_time_days,
            ref_time_days,
        )
        row.update(
            method=method,
            reference_object_id=reference_object_id,
            detected_object_id=detected_object_id,
        )
        rows.append(row)

    finite_rows = [row for row in rows if np.isfinite(row["spatiotemporal_iou"])]
    return (max(finite_rows, key=lambda row: row["spatiotemporal_iou"]) if finite_rows else None), rows


def build_single_reference_match_table(reference_object_id):
    gt_change = _single_object_change_summary("GT", reference_object_id)
    reference_change_volume = gt_change["change_volume"]

    rows = []
    best_matches = {}

    for method in ("M3C2", "TAM3C2"):
        best, _ = _best_detected_match_for_reference(reference_object_id, method)
        best_matches[method] = best

        if best is None:
            rows.append(
                dict(
                    method=method,
                    reference_object_id=reference_object_id,
                    best_detected_object_id=None,
                    spatial_iou_pct=np.nan,
                    temporal_iou_pct=np.nan,
                    spatiotemporal_iou_pct=np.nan,
                    reference_change_volume=reference_change_volume,
                    detected_change_volume=np.nan,
                    detected_reference_change_pct=np.nan,
                    change_difference=np.nan,
                    change_difference_pct_of_ref=np.nan,
                )
            )
            continue

        detected_object_id = int(best["detected_object_id"])
        det_change = _single_object_change_summary(method, detected_object_id)
        detected_change_volume = det_change["change_volume"]
        change_difference = detected_change_volume - reference_change_volume

        rows.append(
            dict(
                method=method,
                reference_object_id=reference_object_id,
                best_detected_object_id=detected_object_id,
                spatial_iou_pct=100.0 * best["spatial_iou"],
                temporal_iou_pct=100.0 * best["temporal_iou"],
                spatiotemporal_iou_pct=100.0 * best["spatiotemporal_iou"],
                reference_change_volume=reference_change_volume,
                detected_change_volume=detected_change_volume,
                detected_reference_change_pct=_safe_percent_ratio(
                    detected_change_volume,
                    reference_change_volume,
                ),
                change_difference=change_difference,
                change_difference_pct_of_ref=_safe_percent_ratio(
                    change_difference,
                    reference_change_volume,
                ),
                shared_corepoints=best["shared_corepoint_count"],
                reference_corepoints=best["reference_corepoint_count"],
                detected_corepoints=best["detected_corepoint_count"],
                overlapping_duration_days=best["overlapping_duration"],
                reference_duration_days=best["reference_duration"],
                detected_duration_days=best["detected_duration"],
            )
        )

    table = pd.DataFrame(rows) if pd is not None else rows
    return table, best_matches


def _object_interval_record(name, object_id):
    timestamps, time_days = _timestamp_info_for(name)
    obj = _analysis_object(name, object_id)
    props = _object_properties(obj, time_days)

    return dict(
        name=name,
        object_id=object_id,
        xy=_object_xy(analyses[name], obj),
        color=OBJECT_COLORS[name],
        interval=dict(
            start_epoch=props["start_epoch"],
            end_epoch=props["end_epoch"],
            start_timestamp=timestamps[props["start_epoch"]],
            end_timestamp=timestamps[props["end_epoch"]],
            start_time_days=props["start_time"],
            end_time_days=props["end_time"],
            duration_days=props["duration"],
        ),
    )


single_match_table, best_matches = build_single_reference_match_table(reference_object_id)

selected_object_ids = {"GT": reference_object_id}
selected_object_ids.update(
    {
        method: None if best_matches[method] is None else int(best_matches[method]["detected_object_id"])
        for method in ("M3C2", "TAM3C2")
    }
)

selected_extent_records = [
    _object_interval_record(name, object_id)
    for name, object_id in selected_object_ids.items()
    if object_id is not None
]

print(f"Selected GT/reference object id: {reference_object_id}")
for method, best in best_matches.items():
    if best is None:
        print(f"{method}: no finite best match")
    else:
        print(
            f"{method}: best object id={int(best['detected_object_id'])}, "
            f"ST-IoU={100.0 * best['spatiotemporal_iou']:.2f}%, "
            f"spatial IoU={100.0 * best['spatial_iou']:.2f}%, "
            f"temporal IoU={100.0 * best['temporal_iou']:.2f}%"
        )

if pd is not None:
    display(single_match_table)
else:
    for row in single_match_table:
        print(row)


### 9.3 Selected-object spatial and temporal plots

Compare temporal intervals, full-scene and zoomed changemaps, and raw distance time series for the selected GT and its two best-overlap detections. Shared axes and colour ranges make method differences directly comparable.


In [ ]:
# Temporal interval figure and separate changemap figures.

def _format_timestamp(ts):
    return f"{ts:%Y-%m-%d}"


def _annotate_interval_endpoint(ax, timestamp, y, text, color, ha, x_offset, y_offset):
    ax.annotate(
        text,
        xy=(timestamp, y),
        xytext=(x_offset, y_offset),
        textcoords="offset points",
        ha=ha,
        va="center",
        fontsize=OBJECT_ANNOTATION_SIZE,
        rotation=35,
        color=color,
        clip_on=False,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=0.5),
    )


def plot_single_reference_temporal_intervals(records):
    fig, ax_time = plt.subplots(
        1,
        1,
        figsize=OBJECT_TEMPORAL_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
        constrained_layout=True,
    )

    y_positions = {"GT": 2, "M3C2": 1, "TAM3C2": 0}
    start_offsets = {"GT": 22, "M3C2": 16, "TAM3C2": 10}
    end_offsets = {"GT": -22, "M3C2": -16, "TAM3C2": -10}

    for record in records:
        name = record["name"]
        interval = record["interval"]
        y = y_positions[name]
        color = record["color"]

        start = interval["start_timestamp"]
        end = interval["end_timestamp"]

        ax_time.hlines(
            y,
            start,
            end,
            color=color,
            linewidth=7,
            alpha=0.75,
            label=f"{name} object {record['object_id']}",
        )
        ax_time.scatter([start, end], [y, y], color=color, s=36, zorder=3)

        _annotate_interval_endpoint(
            ax_time,
            start,
            y,
            _format_timestamp(start),
            color=color,
            ha="left",
            x_offset=5,
            y_offset=start_offsets.get(name, 14),
        )
        _annotate_interval_endpoint(
            ax_time,
            end,
            y,
            _format_timestamp(end),
            color=color,
            ha="right",
            x_offset=-5,
            y_offset=end_offsets.get(name, -14),
        )

    if len(records) >= 2:
        overlap_start = max(record["interval"]["start_timestamp"] for record in records)
        overlap_end = min(record["interval"]["end_timestamp"] for record in records)

        if overlap_start <= overlap_end:
            ax_time.axvspan(
                overlap_start,
                overlap_end,
                color="green",
                alpha=0.15,
                label="common overlap",
            )

    ax_time.set_yticks([0, 1, 2])
    ax_time.set_yticklabels(["TAM3C2", "M3C2", "GT"])
    ax_time.set_ylim(-0.65, 2.65)
    ax_time.set_title("Temporal intervals of selected objects")
    ax_time.set_xlabel("Date")
    ax_time.grid(axis="x", alpha=0.25)
    ax_time.margins(x=0.12)

    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax_time.xaxis.set_major_locator(locator)
    ax_time.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

    for tick in ax_time.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")

    ax_time.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=min(4, len(records) + 1),
        framealpha=0.9,
    )

    _apply_object_plot_font_sizes(fig)
    plt.show()


def plot_single_reference_changemap(records, crange=None):
    background_record = next(
        (record for record in records if record["name"] == "GT"),
        records[0],
    )

    fig, ax_scene, ax_colorbar = _new_single_changemap_figure()

    _plot_spatial_panel(
        ax_scene,
        records,
        background_record=background_record,
        title="Changemap",
        crange=crange,
        zoom=False,
        add_colorbar=True,
        show_legend=True,
        colorbar_ax=ax_colorbar,
    )
    ax_scene.set_box_aspect(1.0)
    ax_scene.set_anchor("C")

    _apply_object_plot_font_sizes(fig)
    plt.show()


def plot_single_reference_zoom(records, crange=None):
    background_record = next(
        (record for record in records if record["name"] == "GT"),
        records[0],
    )

    fig, ax_zoom, ax_colorbar = _new_single_changemap_figure()

    _plot_spatial_panel(
        ax_zoom,
        records,
        background_record=background_record,
        title="Zoom-in objects",
        crange=crange,
        zoom=True,
        add_colorbar=True,
        show_legend=False,
        colorbar_ax=ax_colorbar,
    )
    ax_zoom.set_box_aspect(1.0)
    ax_zoom.set_anchor("C")

    _apply_object_plot_font_sizes(fig)
    plt.show()


plot_single_reference_temporal_intervals(selected_extent_records)
plot_single_reference_changemap(selected_extent_records, crange=None)
plot_single_reference_zoom(selected_extent_records, crange=None)


In [ ]:
# Raw distance time series of the selected GT/M3C2/TAM3C2 objects.
def _distance_series_for_object_plot(st_analysis, use_smoothed=False):
    if use_smoothed:
        raise ValueError(
            "This evaluation is configured to use raw distances only. "
            "Set use_smoothed_time_series=False."
        )
    return np.asarray(st_analysis.distances, dtype=float), "distances"


def _previous_cell_crange(records, crange=None):
    # Use exactly the same changemap scale logic as the previous cell:
    # background_record = GT if available, otherwise records[0].
    background_record = next(
        (record for record in records if record["name"] == "GT"),
        records[0],
    )
    _, magnitudes = _raw_changemap(background_record)
    return _resolve_crange(magnitudes, crange)


def _shared_time_series_ylim_and_ticks(
    records,
    lower_percentile=1.0,
    upper_percentile=99.0,
    pad_ratio=0.08,
    nbins=5,
):
    all_values = []
    protected_values = [0.0]

    for record in records:
        name = record["name"]
        object_id = record["object_id"]

        st_analysis = analyses[name]
        obj = _analysis_object(name, object_id)
        indices = _object_indices(obj)

        distances, _ = _distance_series_for_object_plot(
            st_analysis,
            use_smoothed=use_smoothed_time_series,
        )

        object_series = np.asarray(distances[indices, :], dtype=float)
        finite = object_series[np.isfinite(object_series)]

        if len(finite):
            all_values.append(finite)

        with np.errstate(all="ignore"):
            mean_series = np.nanmean(object_series, axis=0)
        mean_finite = mean_series[np.isfinite(mean_series)]
        if len(mean_finite):
            protected_values.extend(mean_finite.tolist())

    if not all_values:
        return None, None

    all_values = np.concatenate(all_values)

    y_min = float(np.nanpercentile(all_values, lower_percentile))
    y_max = float(np.nanpercentile(all_values, upper_percentile))

    protected_values = np.asarray(protected_values, dtype=float)
    protected_values = protected_values[np.isfinite(protected_values)]

    if len(protected_values):
        y_min = min(y_min, float(np.nanmin(protected_values)))
        y_max = max(y_max, float(np.nanmax(protected_values)))

    if y_min == y_max:
        pad = max(abs(y_min) * pad_ratio, 0.1)
    else:
        pad = pad_ratio * (y_max - y_min)

    y_min -= pad
    y_max += pad

    locator = MaxNLocator(nbins=nbins)
    ticks = locator.tick_values(y_min, y_max)

    return (float(ticks[0]), float(ticks[-1])), ticks


def _plot_single_method_object_time_series_figure(
    record,
    shared_crange,
    shared_ylim,
    shared_yticks,
):
    name = record["name"]
    object_id = record["object_id"]

    fig, ax_ts, ax_map, ax_colorbar = _new_time_series_changemap_figure()

    st_analysis = analyses[name]
    obj = _analysis_object(name, object_id)
    indices = _object_indices(obj)

    timestamps, _ = _timestamp_info_for(name)
    distances, distance_label = _distance_series_for_object_plot(
        st_analysis,
        use_smoothed=use_smoothed_time_series,
    )

    interval = record["interval"]
    start_epoch = int(interval["start_epoch"])
    end_epoch = int(interval["end_epoch"])

    object_series = distances[indices, :]
    changemap = distances[:, end_epoch] - distances[:, start_epoch]

    cmap = plt.get_cmap("seismic_r").copy()
    norm = plt.Normalize(vmin=-shared_crange, vmax=shared_crange)

    for idx, series in zip(indices, object_series):
        ax_ts.plot(
            timestamps,
            series,
            color=cmap(norm(changemap[idx])),
            alpha=0.22,
            linewidth=0.7,
        )

    with np.errstate(all="ignore"):
        mean_series = np.nanmean(object_series, axis=0)

    ax_ts.plot(
        timestamps,
        mean_series,
        color="black",
        linewidth=2.0,
        label="mean object corepoint time series",
    )
    ax_ts.axvspan(
        interval["start_timestamp"],
        interval["end_timestamp"],
        color="grey",
        alpha=0.22,
        label="4D-OBC timespan",
    )
    ax_ts.axhline(0, color="0.5", linewidth=0.8, linestyle="--")

    if shared_ylim is not None:
        ax_ts.set_ylim(*shared_ylim)
        ax_ts.set_yticks(shared_yticks)

    ax_ts.set_xlabel("Date")
    ax_ts.set_ylabel("Distance [m]")
    ax_ts.set_title(
        f"{name} object {object_id}: "
        f"{len(indices)} corepoints, raw {distance_label}, "
        f"{interval['duration_days']:.1f} days"
    )
    ax_ts.grid(alpha=0.25)
    ax_ts.legend(loc="upper right", framealpha=0.9)

    for tick in ax_ts.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")
    ax_ts.tick_params(axis="both", which="both", labelbottom=True, labelleft=True)

    _plot_spatial_panel(
        ax_map,
        [record],
        background_record=record,
        title=f"{name} changemap",
        crange=shared_crange,
        zoom=False,
        add_colorbar=True,
        show_legend=True,
        colorbar_ax=ax_colorbar,
    )

    ax_map.set_box_aspect(1.0)
    ax_map.set_anchor("C")
    ax_map.tick_params(axis="both", which="both", labelbottom=True, labelleft=True)

    _apply_object_plot_font_sizes(fig)
    plt.show()


def plot_single_reference_object_time_series(records, crange=None):
    shared_crange = _previous_cell_crange(records, crange=crange)
    shared_ylim, shared_yticks = _shared_time_series_ylim_and_ticks(records)

    print(f"Using previous-cell changemap scale for all three panels: +/-{shared_crange:.3f} m")
    if shared_ylim is not None:
        print(
            f"Using robust shared time-series y-axis: "
            f"{shared_ylim[0]:.3f} to {shared_ylim[1]:.3f} m"
        )

    record_by_name = {record["name"]: record for record in records}

    for name in OBJECT_METHODS:
        record = record_by_name.get(name)
        if record is None:
            fig, ax = plt.subplots(
                1,
                1,
                figsize=OBJECT_TIME_SERIES_MAP_FIGSIZE,
                dpi=OBJECT_FIG_DPI,
                constrained_layout=True,
            )
            ax.text(
                0.5,
                0.5,
                f"{name}: no matched object",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=OBJECT_FONT_SIZE,
            )
            ax.set_axis_off()
            _apply_object_plot_font_sizes(fig)
            plt.show()
            continue

        _plot_single_method_object_time_series_figure(
            record,
            shared_crange,
            shared_ylim,
            shared_yticks,
        )


plot_single_reference_object_time_series(selected_extent_records, crange=None)


### 9.4 selected-reference object- fragments spatial temporal plots

In [ ]:
individual_reference_object_id=6

In [ ]:
# Investigate all detected objects with non-zero ST-IoU
# against one selected reference object.
minimum_st_iou=IOU_EPS

gt_objects=_objects_for_analysis(analyses["GT"])
if not 0<=individual_reference_object_id<len(gt_objects):
    raise IndexError(
        f"Reference index {individual_reference_object_id} outside "
        f"[0,{len(gt_objects)-1}]"
    )

individual_overlap_df=pairwise_metrics_df.loc[
    (pairwise_metrics_df["reference_object_index"]
     ==individual_reference_object_id)
    &(pairwise_metrics_df["spatiotemporal_iou"].fillna(0)
      >minimum_st_iou)
].copy()

method_order={"M3C2":0,"TAM3C2":1}
individual_overlap_df["_method_order"]=(
    individual_overlap_df["method"].map(method_order)
)
individual_overlap_df=(
    individual_overlap_df
    .sort_values(
        ["_method_order","spatiotemporal_iou"],
        ascending=[True,False],
    )
    .drop(columns="_method_order")
    .reset_index(drop=True)
)

if individual_overlap_df.empty:
    raise ValueError(
        f"No detected objects have non-zero ST-IoU with "
        f"GT object {individual_reference_object_id}."
    )

individual_overlap_table=individual_overlap_df[[
    "method",
    "reference_object_index",
    "detected_object_index",
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
    "shared_corepoint_count",
    "overlapping_duration",
]].copy()

individual_overlap_table[[
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
]]*=100

individual_overlap_table=individual_overlap_table.rename(columns={
    "spatial_iou":"spatial_iou_pct",
    "temporal_iou":"temporal_iou_pct",
    "spatiotemporal_iou":"spatiotemporal_iou_pct",
    "overlapping_duration":"overlapping_duration_days",
})

print(
    f"GT object {individual_reference_object_id}: "
    f"{len(individual_overlap_table)} detected objects "
    f"with non-zero ST-IoU"
)
display(individual_overlap_table.round({
    "spatial_iou_pct":2,
    "temporal_iou_pct":2,
    "spatiotemporal_iou_pct":2,
    "overlapping_duration_days":2,
}))


all_overlap_records=[
    _object_interval_record(
        "GT",
        individual_reference_object_id,
    )
]

for row in individual_overlap_df.itertuples(index=False):
    record=_object_interval_record(
        row.method,
        int(row.detected_object_index),
    )
    record.update({
        "spatial_iou":float(row.spatial_iou),
        "temporal_iou":float(row.temporal_iou),
        "spatiotemporal_iou":float(row.spatiotemporal_iou),
    })
    all_overlap_records.append(record)


def plot_all_overlap_temporal_intervals(records):
    fig,ax=plt.subplots(
        figsize=OBJECT_TEMPORAL_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
        constrained_layout=True,
    )

    gt_record=records[0]
    gt_start=gt_record["interval"]["start_timestamp"]
    gt_end=gt_record["interval"]["end_timestamp"]

    ax.axvspan(
        gt_start,
        gt_end,
        color="green",
        alpha=0.12,
        label="GT temporal extent",
        zorder=0,
    )

    y_positions=np.arange(len(records))[::-1]
    y_labels=[]

    for row_idx,(record,y) in enumerate(
        zip(records,y_positions)
    ):
        name=record["name"]
        object_id=record["object_id"]
        interval=record["interval"]
        start=interval["start_timestamp"]
        end=interval["end_timestamp"]
        color=record["color"]

        ax.hlines(
            y,start,end,
            color=color,
            linewidth=7 if name=="GT" else 5,
            alpha=0.78,
            zorder=2,
        )
        ax.scatter(
            [start,end],[y,y],
            color=color,
            s=40 if name=="GT" else 30,
            zorder=3,
        )

        _annotate_interval_endpoint(
            ax,start,y,
            _format_timestamp(start),
            color=color,
            ha="left",
            x_offset=5,
            y_offset=10,
        )
        _annotate_interval_endpoint(
            ax,end,y,
            _format_timestamp(end),
            color=color,
            ha="right",
            x_offset=-5,
            y_offset=-10,
        )

        if name=="GT":
            y_labels.append(f"GT object {object_id}")
        else:
            y_labels.append(
                f"{name} object {object_id} "
                f"(ST={100*record['spatiotemporal_iou']:.1f}%)"
            )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.set_ylim(-0.8,len(records)-0.2)
    ax.set_title(
        f"Temporal intervals overlapping GT object "
        f"{individual_reference_object_id}"
    )
    ax.set_xlabel("Date")
    ax.grid(axis="x",alpha=0.25)
    ax.margins(x=0.12)

    locator=mdates.AutoDateLocator(
        minticks=4,
        maxticks=8,
    )
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    for tick in ax.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")

    ax.plot(
        [],[],color=OBJECT_COLORS["GT"],
        linewidth=6,label="GT",
    )
    ax.plot(
        [],[],color=OBJECT_COLORS["M3C2"],
        linewidth=6,label="M3C2",
    )
    ax.plot(
        [],[],color=OBJECT_COLORS["TAM3C2"],
        linewidth=6,label="TAM3C2",
    )

    handles,labels=ax.get_legend_handles_labels()
    ax.legend(
        handles,labels,
        loc="upper center",
        bbox_to_anchor=(0.5,-0.22),
        ncol=4,
        framealpha=0.9,
    )

    _apply_object_plot_font_sizes(fig)
    plt.show()


def _annotate_spatial_object_ids(ax,records):
    for record in records:
        xy=np.asarray(record["xy"],dtype=float)
        if len(xy)==0:
            continue

        center=np.nanmean(xy,axis=0)
        label=(
            f"GT {record['object_id']}"
            if record["name"]=="GT"
            else f"{record['name']} {record['object_id']}"
        )

        ax.annotate(
            label,
            xy=center,
            xytext=(4,4),
            textcoords="offset points",
            color=record["color"],
            fontsize=OBJECT_ANNOTATION_SIZE,
            fontweight="bold",
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.72,
                pad=0.5,
            ),
            zorder=10,
        )


def plot_all_overlap_changemap(records,zoom=False,crange=None):
    background_record=records[0]
    fig,ax_scene,ax_colorbar=_new_single_changemap_figure()

    _plot_spatial_panel(
        ax_scene,
        records,
        background_record=background_record,
        title=(
            f"Zoom-in objects for GT "
            f"{individual_reference_object_id}"
            if zoom
            else f"Changemap for GT "
                 f"{individual_reference_object_id}"
        ),
        crange=crange,
        zoom=zoom,
        add_colorbar=True,
        show_legend=not zoom,
        colorbar_ax=ax_colorbar,
    )

    _annotate_spatial_object_ids(
        ax_scene,
        records,
    )
    ax_scene.set_box_aspect(1.0)
    ax_scene.set_anchor("C")
    _apply_object_plot_font_sizes(fig)
    plt.show()


plot_all_overlap_temporal_intervals(
    all_overlap_records
)
plot_all_overlap_changemap(
    all_overlap_records,
    zoom=False,
    crange=None,
)
plot_all_overlap_changemap(
    all_overlap_records,
    zoom=True,
    crange=None,
)

In [ ]:
# Plot one time-series and spatial-convex-hull figure for every object.
# All time-series panels use the same distance y-axis.

MAX_INDIVIDUAL_SERIES=300

# ------------------------------------------------------------------
# Calculate a common zero-centred distance y-axis
# ------------------------------------------------------------------

all_selected_distances=[]

for record in all_overlap_records:
    name=record["name"]
    object_id=int(record["object_id"])
    obj=_analysis_object(name,object_id)
    indices=_object_indices(obj)

    values=np.asarray(
        analyses[name].distances,
        dtype=float,
    )[indices,:]

    finite_values=values[np.isfinite(values)]
    if len(finite_values):
        all_selected_distances.append(finite_values)

if not all_selected_distances:
    raise ValueError(
        "No finite distances found for the selected objects."
    )

all_selected_distances=np.concatenate(
    all_selected_distances
)

maximum_absolute_distance=float(
    np.max(np.abs(all_selected_distances))
)

distance_padding=max(
    0.05*maximum_absolute_distance,
    1e-6,
)

COMMON_DISTANCE_YLIM=(
    -maximum_absolute_distance-distance_padding,
    maximum_absolute_distance+distance_padding,
)

print(
    "Common distance y-axis: "
    f"{COMMON_DISTANCE_YLIM[0]:.3f} to "
    f"{COMMON_DISTANCE_YLIM[1]:.3f} m"
)


# ------------------------------------------------------------------
# Plot one object
# ------------------------------------------------------------------

def plot_object_time_series_and_spatial_hull(
    record,
    gt_record,
    distance_ylim,
    crange=None,
):
    name=record["name"]
    object_id=int(record["object_id"])
    color=record["color"]

    obj=_analysis_object(name,object_id)
    indices=_object_indices(obj)

    timestamps,_=_timestamp_info_for(name)
    timestamps=np.asarray(timestamps,dtype=object)

    distances=np.asarray(
        analyses[name].distances,
        dtype=float,
    )

    object_series=distances[indices,:]
    valid_rows=np.any(
        np.isfinite(object_series),
        axis=1,
    )
    object_series=object_series[valid_rows]

    if len(object_series)>MAX_INDIVIDUAL_SERIES:
        selected_rows=np.linspace(
            0,
            len(object_series)-1,
            MAX_INDIVIDUAL_SERIES,
            dtype=int,
        )
        displayed_series=object_series[selected_rows]
    else:
        displayed_series=object_series

    fig,ax_time,ax_map,ax_colorbar=(
        _new_time_series_changemap_figure()
    )

    # --------------------------------------------------------------
    # Raw distance time series
    # --------------------------------------------------------------

    for series in displayed_series:
        ax_time.plot(
            timestamps,
            series,
            color=color,
            linewidth=0.6,
            alpha=0.12,
            zorder=1,
        )

    if len(object_series):
        median_series=np.nanmedian(
            object_series,
            axis=0,
        )
        lower_series=np.nanpercentile(
            object_series,
            25,
            axis=0,
        )
        upper_series=np.nanpercentile(
            object_series,
            75,
            axis=0,
        )

        ax_time.fill_between(
            timestamps,
            lower_series,
            upper_series,
            color=color,
            alpha=0.18,
            label="25–75% range",
            zorder=2,
        )

        ax_time.plot(
            timestamps,
            median_series,
            color=color,
            linewidth=2.8,
            label="Median distance",
            zorder=3,
        )

    start=record["interval"]["start_timestamp"]
    end=record["interval"]["end_timestamp"]

    ax_time.axvspan(
        start,
        end,
        color=color,
        alpha=0.10,
        label="Object temporal interval",
        zorder=0,
    )

    ax_time.axvline(
        start,
        color=color,
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
    )

    ax_time.axvline(
        end,
        color=color,
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
    )

    ax_time.axhline(
        0,
        color="black",
        linewidth=0.8,
        alpha=0.45,
    )

    if name=="GT":
        title=(
            f"GT object {object_id}: "
            f"raw distance time series"
        )
    else:
        st_iou=100*record["spatiotemporal_iou"]
        title=(
            f"{name} object {object_id}: "
            f"raw distance time series "
            f"(ST-IoU={st_iou:.1f}%)"
        )

    ax_time.set_title(title)
    ax_time.set_xlabel("Date")
    ax_time.set_ylabel("Distance [m]")
    ax_time.set_ylim(distance_ylim)
    ax_time.grid(alpha=0.25)

    locator=mdates.AutoDateLocator(
        minticks=5,
        maxticks=10,
    )
    ax_time.xaxis.set_major_locator(locator)
    ax_time.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    for tick in ax_time.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")

    ax_time.legend(
        loc="upper center",
        bbox_to_anchor=(0.5,-0.18),
        ncol=3,
        framealpha=0.9,
    )

    # --------------------------------------------------------------
    # Spatial convex hull
    # --------------------------------------------------------------

    if name=="GT":
        spatial_records=[record]
    else:
        spatial_records=[gt_record,record]

    _plot_spatial_panel(
        ax_map,
        spatial_records,
        background_record=record,
        title=(
            f"Spatial convex hull: "
            f"{name} object {object_id}"
        ),
        crange=crange,
        zoom=True,
        add_colorbar=True,
        show_legend=True,
        legend_loc="upper right",
        colorbar_ax=ax_colorbar,
    )

    for spatial_record in spatial_records:
        xy=np.asarray(
            spatial_record["xy"],
            dtype=float,
        )

        if len(xy)==0:
            continue

        center=np.nanmean(xy,axis=0)

        ax_map.annotate(
            (
                f"{spatial_record['name']} "
                f"{spatial_record['object_id']}"
            ),
            xy=center,
            xytext=(4,4),
            textcoords="offset points",
            color=spatial_record["color"],
            fontweight="bold",
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.72,
                pad=0.5,
            ),
            zorder=10,
        )

    ax_map.set_box_aspect(1.0)
    ax_map.set_anchor("C")

    _apply_object_plot_font_sizes(fig)
    plt.show()


# ------------------------------------------------------------------
# Plot GT and every detected object with non-zero ST-IoU
# ------------------------------------------------------------------

gt_overlap_record=next(
    record
    for record in all_overlap_records
    if record["name"]=="GT"
)

for object_record in all_overlap_records:
    print(
        f"Plotting {object_record['name']} "
        f"object {object_record['object_id']}"
    )

    plot_object_time_series_and_spatial_hull(
        object_record,
        gt_overlap_record,
        distance_ylim=COMMON_DISTANCE_YLIM,
        crange=None,
    )

### 9.5 zero ST-IOU match spatial and  temporal plot
给定指定方法的detected object index，画出存在spatial或者temporal overlap的reference object图

In [ ]:
zero_case_method="TAM3C2"
zero_case_detected_index=11

In [ ]:
# Individual diagnostic plot for one zero-ST-IoU detected object.
MAX_TIME_SERIES_LINES=300
FIXED_DISTANCE_YLIM=None

if zero_case_method not in METHODS:
    raise ValueError(
        f"zero_case_method must be one of {METHODS}"
    )

zero_case_rows=zero_st_overlap_detail_df.loc[
    (zero_st_overlap_detail_df["method"]==zero_case_method)
    &(zero_st_overlap_detail_df["detected_object_index"]
      ==zero_case_detected_index)
]

if zero_case_rows.empty:
    raise ValueError(
        f"{zero_case_method} object {zero_case_detected_index} "
        "is not classified as a zero-ST-overlap object."
    )

zero_case_reason=zero_case_rows.iloc[0]["zero_st_reason"]

detected_pairs=pairwise_metrics_df.loc[
    (pairwise_metrics_df["method"]==zero_case_method)
    &(pairwise_metrics_df["detected_object_index"]
      ==zero_case_detected_index)
].copy()

maximum_st_iou=float(
    detected_pairs["spatiotemporal_iou"]
    .fillna(0)
    .max()
)

if maximum_st_iou>IOU_EPS:
    raise ValueError(
        f"Selected object has maximum ST-IoU "
        f"{maximum_st_iou:.6f}, so it is not a zero-ST-IoU case."
    )

spatial_values=detected_pairs["spatial_iou"].fillna(0)
temporal_values=detected_pairs["temporal_iou"].fillna(0)

best_spatial_row=(
    detected_pairs.loc[spatial_values.idxmax()]
    if spatial_values.max()>IOU_EPS
    else None
)

best_temporal_row=(
    detected_pairs.loc[temporal_values.idxmax()]
    if temporal_values.max()>IOU_EPS
    else None
)

reference_candidates={}

if best_spatial_row is not None:
    ref_idx=int(
        best_spatial_row["reference_object_index"]
    )
    reference_candidates[ref_idx]={
        "roles":["best spatial overlap"],
        "color":"black",
    }

if best_temporal_row is not None:
    ref_idx=int(
        best_temporal_row["reference_object_index"]
    )
    if ref_idx in reference_candidates:
        reference_candidates[ref_idx]["roles"].append(
            "best temporal overlap"
        )
    else:
        reference_candidates[ref_idx]={
            "roles":["best temporal overlap"],
            "color":"black",
        }


def build_zero_case_record(
    name,
    object_index,
    label,
    color,
):
    timestamps,time_days=timestamp_info[name]
    obj=analyses[name].objects[object_index]
    props=_object_properties(obj,time_days)
    indices=np.asarray(obj.indices,dtype=int)
    xy=np.asarray(
        analyses[name].corepoints.cloud
    )[indices,:2]

    return {
        "name":name,
        "object_id":int(object_index),
        "label":label,
        "color":color,
        "object":obj,
        "indices":indices,
        "xy":xy,
        "timestamps":np.asarray(
            timestamps,
            dtype=object,
        ),
        "start_epoch":props["start_epoch"],
        "end_epoch":props["end_epoch"],
        "start_timestamp":timestamps[
            props["start_epoch"]
        ],
        "end_timestamp":timestamps[
            props["end_epoch"]
        ],
    }


zero_case_records=[
    build_zero_case_record(
        zero_case_method,
        zero_case_detected_index,
        (
            f"{zero_case_method} object "
            f"{zero_case_detected_index}"
        ),
        METHOD_COLORS[zero_case_method],
    )
]

for reference_index,candidate in reference_candidates.items():
    role=" + ".join(candidate["roles"])
    zero_case_records.append(
        build_zero_case_record(
            "GT",
            reference_index,
            f"GT object {reference_index} ({role})",
            candidate["color"],
        )
    )

diagnostic_rows=[]

for reference_index,candidate in reference_candidates.items():
    pair=detected_pairs.loc[
        detected_pairs["reference_object_index"]
        ==reference_index
    ].iloc[0]

    diagnostic_rows.append({
        "detected_method":zero_case_method,
        "detected_object_index":zero_case_detected_index,
        "reference_role":" + ".join(candidate["roles"]),
        "reference_object_index":reference_index,
        "spatial_iou":float(pair["spatial_iou"]),
        "temporal_iou":float(pair["temporal_iou"]),
        "spatiotemporal_iou":float(
            pair["spatiotemporal_iou"]
        ),
    })

diagnostic_candidates_df=pd.DataFrame(
    diagnostic_rows,
    columns=[
        "detected_method",
        "detected_object_index",
        "reference_role",
        "reference_object_index",
        "spatial_iou",
        "temporal_iou",
        "spatiotemporal_iou",
    ],
)

print(
    f"Selected zero-ST case: {zero_case_method} "
    f"object {zero_case_detected_index}"
)
print(f"Diagnostic category: {zero_case_reason}")

if diagnostic_candidates_df.empty:
    print(
        "No spatial or temporal reference candidate exists. "
        "Only the detected object will be plotted."
    )
else:
    display(diagnostic_candidates_df.round({
        "spatial_iou":3,
        "temporal_iou":3,
        "spatiotemporal_iou":3,
    }))


# --------------------------------------------------------------
# Common distance y-axis for every selected object
# --------------------------------------------------------------

all_case_distances=[]

for record in zero_case_records:
    values=np.asarray(
        analyses[record["name"]].distances,
        dtype=float,
    )[record["indices"],:]

    finite_values=values[np.isfinite(values)]
    if len(finite_values):
        all_case_distances.append(finite_values)

if not all_case_distances:
    raise ValueError(
        "No finite distance values found for this case."
    )

all_case_distances=np.concatenate(all_case_distances)

if FIXED_DISTANCE_YLIM is None:
    maximum_absolute_distance=float(
        np.max(np.abs(all_case_distances))
    )
    distance_padding=max(
        0.05*maximum_absolute_distance,
        1e-6,
    )
    ZERO_CASE_DISTANCE_YLIM=(
        -maximum_absolute_distance-distance_padding,
        maximum_absolute_distance+distance_padding,
    )
else:
    ZERO_CASE_DISTANCE_YLIM=FIXED_DISTANCE_YLIM

print(
    "Common distance y-axis: "
    f"{ZERO_CASE_DISTANCE_YLIM[0]:.3f} to "
    f"{ZERO_CASE_DISTANCE_YLIM[1]:.3f} m"
)


# --------------------------------------------------------------
# Common changemap colour range
# --------------------------------------------------------------

case_change_magnitudes=[]

for record in zero_case_records:
    distances=np.asarray(
        analyses[record["name"]].distances,
        dtype=float,
    )
    magnitudes=(
        distances[:,record["end_epoch"]]
        -distances[:,record["start_epoch"]]
    )
    finite=magnitudes[np.isfinite(magnitudes)]
    if len(finite):
        case_change_magnitudes.append(finite)

if case_change_magnitudes:
    ZERO_CASE_CRANGE=float(
        np.max(
            np.abs(
                np.concatenate(case_change_magnitudes)
            )
        )
    )
else:
    ZERO_CASE_CRANGE=1.0

if ZERO_CASE_CRANGE<=0:
    ZERO_CASE_CRANGE=1.0


def draw_zero_case_hull(
    ax,
    record,
):
    xy=np.asarray(record["xy"],dtype=float)
    if len(xy)==0:
        return

    unique_xy=np.unique(xy,axis=0)

    ax.scatter(
        xy[:,0],
        xy[:,1],
        color=record["color"],
        s=12,
        alpha=0.45,
        linewidths=0,
        zorder=4,
    )

    if len(unique_xy)>=3:
        try:
            hull=ConvexHull(unique_xy)
            hull_xy=unique_xy[hull.vertices]

            ax.add_patch(
                Polygon(
                    hull_xy,
                    closed=True,
                    facecolor=record["color"],
                    edgecolor=record["color"],
                    linewidth=2.4,
                    alpha=0.06,
                    zorder=5,
                )
            )

            ax.plot(
                np.r_[hull_xy[:,0],hull_xy[0,0]],
                np.r_[hull_xy[:,1],hull_xy[0,1]],
                color=record["color"],
                linewidth=2.4,
                label=record["label"],
                zorder=6,
            )
        except QhullError:
            ax.plot(
                unique_xy[:,0],
                unique_xy[:,1],
                color=record["color"],
                linewidth=2.4,
                label=record["label"],
                zorder=6,
            )
    else:
        ax.plot(
            unique_xy[:,0],
            unique_xy[:,1],
            color=record["color"],
            linewidth=2.4,
            marker="o",
            label=record["label"],
            zorder=6,
        )

    center=np.nanmean(xy,axis=0)
    ax.annotate(
        record["label"],
        xy=center,
        xytext=(4,4),
        textcoords="offset points",
        color=record["color"],
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.72,
            pad=0.5,
        ),
        zorder=8,
    )


def plot_zero_case_object(
    record,
    all_records,
):
    name=record["name"]
    object_index=record["object_id"]
    color=record["color"]

    distances=np.asarray(
        analyses[name].distances,
        dtype=float,
    )
    object_series=distances[
        record["indices"],:
    ]

    valid_rows=np.any(
        np.isfinite(object_series),
        axis=1,
    )
    object_series=object_series[valid_rows]

    if len(object_series)>MAX_TIME_SERIES_LINES:
        selected_rows=np.linspace(
            0,
            len(object_series)-1,
            MAX_TIME_SERIES_LINES,
            dtype=int,
        )
        displayed_series=object_series[selected_rows]
    else:
        displayed_series=object_series

    fig=plt.figure(
        figsize=(16.7,6.2),
        dpi=120,
        constrained_layout=True,
    )
    grid=fig.add_gridspec(
        1,
        3,
        width_ratios=[9.6,4.8,0.25],
    )

    ax_time=fig.add_subplot(grid[0,0])
    ax_map=fig.add_subplot(grid[0,1])
    ax_colorbar=fig.add_subplot(grid[0,2])

    for series in displayed_series:
        ax_time.plot(
            record["timestamps"],
            series,
            color=color,
            linewidth=0.6,
            alpha=0.10,
            zorder=1,
        )

    if len(object_series):
        median_series=np.nanmedian(
            object_series,
            axis=0,
        )
        lower_series=np.nanpercentile(
            object_series,
            25,
            axis=0,
        )
        upper_series=np.nanpercentile(
            object_series,
            75,
            axis=0,
        )

        ax_time.fill_between(
            record["timestamps"],
            lower_series,
            upper_series,
            color=color,
            alpha=0.18,
            label="25–75% range",
            zorder=2,
        )

        ax_time.plot(
            record["timestamps"],
            median_series,
            color=color,
            linewidth=2.8,
            label="Median distance",
            zorder=3,
        )

    ax_time.axvspan(
        record["start_timestamp"],
        record["end_timestamp"],
        color=color,
        alpha=0.10,
        label="Object temporal interval",
        zorder=0,
    )

    ax_time.axvline(
        record["start_timestamp"],
        color=color,
        linestyle="--",
        linewidth=1.2,
    )
    ax_time.axvline(
        record["end_timestamp"],
        color=color,
        linestyle="--",
        linewidth=1.2,
    )
    ax_time.axhline(
        0,
        color="black",
        linewidth=0.8,
        alpha=0.45,
    )

    ax_time.set(
        title=f"{record['label']}: raw distance time series",
        xlabel="Date",
        ylabel="Distance [m]",
        ylim=ZERO_CASE_DISTANCE_YLIM,
    )
    ax_time.grid(alpha=0.25)

    locator=mdates.AutoDateLocator(
        minticks=5,
        maxticks=10,
    )
    ax_time.xaxis.set_major_locator(locator)
    ax_time.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    for tick in ax_time.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")

    ax_time.legend(
        loc="upper center",
        bbox_to_anchor=(0.5,-0.18),
        ncol=3,
        framealpha=0.9,
    )

    cloud=np.asarray(
        analyses[name].corepoints.cloud
    )
    magnitudes=(
        distances[:,record["end_epoch"]]
        -distances[:,record["start_epoch"]]
    )

    scatter=ax_map.scatter(
        cloud[:,0],
        cloud[:,1],
        c=magnitudes,
        cmap="seismic_r",
        vmin=-ZERO_CASE_CRANGE,
        vmax=ZERO_CASE_CRANGE,
        s=1,
        linewidths=0,
        alpha=0.85,
        zorder=1,
    )

    for spatial_record in all_records:
        draw_zero_case_hull(
            ax_map,
            spatial_record,
        )

    all_xy=np.vstack([
        spatial_record["xy"]
        for spatial_record in all_records
        if len(spatial_record["xy"])
    ])

    x_min,x_max=np.nanmin(all_xy[:,0]),np.nanmax(all_xy[:,0])
    y_min,y_max=np.nanmin(all_xy[:,1]),np.nanmax(all_xy[:,1])

    x_pad=max(0.08*(x_max-x_min),0.5)
    y_pad=max(0.08*(y_max-y_min),0.5)

    ax_map.set_xlim(
        x_min-x_pad,
        x_max+x_pad,
    )
    ax_map.set_ylim(
        y_min-y_pad,
        y_max+y_pad,
    )
    ax_map.set(
        title=f"Spatial convex hull: {record['label']}",
        xlabel="X [m]",
        ylabel="Y [m]",
    )
    ax_map.set_aspect(
        "equal",
        adjustable="box",
    )
    ax_map.grid(alpha=0.25)
    ax_map.legend(
        loc="upper right",
        framealpha=0.9,
    )

    colorbar=fig.colorbar(
        scatter,
        cax=ax_colorbar,
        format="%.2f",
    )
    colorbar.set_label(
        "Change magnitude [m], "
        f"scale +/-{ZERO_CASE_CRANGE:.2f}"
    )

    plt.show()


for record in zero_case_records:
    print(f"Plotting {record['label']}")
    plot_zero_case_object(
        record,
        zero_case_records,
    )